#  Pipeline



##  Importing libraries

In [1]:
import os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── RDKit 
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, QED
from rdkit import RDConfig
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.rdMolDescriptors import (
    CalcTPSA, CalcNumHBD, CalcNumHBA,
    CalcNumRotatableBonds, CalcNumAromaticRings,
    CalcFractionCSP3, CalcNumRings,
    CalcNumSpiroAtoms, CalcNumAtomStereoCenters,
)

# ── ML 
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold

# ── LightGBM 
try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
    print('LightGBM available ✓')
except ImportError:
    HAS_LGB = False
    print('LightGBM not found — ensemble will use XGB + RF only')

# ── CatBoost (special ensemble for NR-AR / NR-ER) 
try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
    print('CatBoost available ✓')
except ImportError:
    HAS_CAT = False
    print('CatBoost not found — install with: pip install catboost')

# ── BalancedRandomForest (imbalanced-learn) 
try:
    from imblearn.ensemble import BalancedRandomForestClassifier
    HAS_BRF = True
    print('BalancedRandomForest available ✓')
except ImportError:
    HAS_BRF = False
    print('imbalanced-learn not found — install with: pip install imbalanced-learn')


SPECIAL_ENDPOINTS  = {'NR-AR', 'NR-AR-LBD', 'NR-ER'}
RF_STRONG_ENDPOINTS = {'NR-PPAR-gamma', 'SR-ATAD5', 'SR-HSE', 'NR-ER-LBD'}
RF_WEAK_ENDPOINTS = {'NR-AhR', 'NR-Aromatase', 'SR-ARE', 'SR-MMP', 'SR-p53'}

print(f'Special  endpoints (CAT+BRF)  : {SPECIAL_ENDPOINTS}')
print(f'RF-strong endpoints (RF=0.5)  : {RF_STRONG_ENDPOINTS}')
print(f'RF-weak   endpoints (XGB=0.6) : {RF_WEAK_ENDPOINTS}')

for d in ['eda_plots', 'models', 'results', 'plots']:
    os.makedirs(d, exist_ok=True)

plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

COLORS = ['#378ADD','#639922','#D85A30','#534AB7',
          '#0F6E56','#BA7517','#A32D2D','#185FA5',
          '#7F77DD','#1D9E75','#E24B4A','#EF9F27']

print('All imports OK')


LightGBM available ✓
CatBoost available ✓
BalancedRandomForest available ✓
Special  endpoints (CAT+BRF)  : {'NR-ER', 'NR-AR', 'NR-AR-LBD'}
RF-strong endpoints (RF=0.5)  : {'SR-ATAD5', 'NR-PPAR-gamma', 'NR-ER-LBD', 'SR-HSE'}
RF-weak   endpoints (XGB=0.6) : {'SR-MMP', 'SR-ARE', 'SR-p53', 'NR-Aromatase', 'NR-AhR'}
All imports OK


## Tox21 endpoint names

In [2]:
TOX21_ENDPOINTS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]
print('Endpoints defined:', len(TOX21_ENDPOINTS))


Endpoints defined: 12


##  Loading data 

In [5]:
df = pd.read_csv('tox21.csv')

smiles_col = next(
    (c for c in df.columns
     if c.lower() in ['smiles', 'mol', 'structure']),
    None
)
if smiles_col is None:
    raise ValueError('No SMILES column found in tox21.csv')
if smiles_col != 'smiles':
    df = df.rename(columns={smiles_col: 'smiles'})

toxicity_labels = [e for e in TOX21_ENDPOINTS if e in df.columns]

print(f'Rows             : {len(df):,}')
print(f'Columns          : {df.shape[1]}')
print(f'Toxicity labels  : {len(toxicity_labels)}/12')
print(f'Labels found     : {toxicity_labels}')


Rows             : 7,831
Columns          : 14
Toxicity labels  : 12/12
Labels found     : ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


## Parsing SMILES

In [6]:
mols         = []
valid_rows   = []
invalid_rows = []

for i, smi in enumerate(df['smiles']):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is not None:
        mols.append(mol)
        valid_rows.append(i)
    else:
        invalid_rows.append(i)

df = df.iloc[valid_rows].reset_index(drop=True)

print(f'Valid molecules  : {len(mols):,}')
print(f'Dropped          : {len(invalid_rows)} (invalid SMILES)')
print(f'Working dataset  : {len(df):,} compounds')


[01:05:11] WARNING: not removing hydrogen atom without neighbors
[01:05:11] Explicit valence for atom # 8 Al, 6, is greater than permitted
[01:05:12] Explicit valence for atom # 3 Al, 6, is greater than permitted
[01:05:12] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:05:12] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:05:12] Explicit valence for atom # 9 Al, 6, is greater than permitted
[01:05:12] Explicit valence for atom # 5 Al, 6, is greater than permitted
[01:05:13] Explicit valence for atom # 16 Al, 6, is greater than permitted
[01:05:13] Explicit valence for atom # 20 Al, 6, is greater than permitted


Valid molecules  : 7,823
Dropped          : 8 (invalid SMILES)
Working dataset  : 7,823 compounds



## EDA — Class Balance (required for scale_pos_weight)

In [7]:
from rdkit.Chem import Descriptors

balance_stats = []
for ep in toxicity_labels:
    col     = df[ep].dropna()
    n_total = len(col)
    n_toxic = int(col.sum())
    n_safe  = n_total - n_toxic
    pct     = 100 * n_toxic / n_total if n_total > 0 else 0
    ratio   = round(n_safe / n_toxic, 1) if n_toxic > 0 else 9999
    balance_stats.append({
        'endpoint': ep, 'n_total': n_total,
        'n_toxic': n_toxic, 'n_safe': n_safe,
        'pct_toxic': round(pct, 1),
        'scale_pos_weight': ratio
    })

bal_df  = pd.DataFrame(balance_stats)
spw_dict = dict(zip(bal_df['endpoint'], bal_df['scale_pos_weight']))

print(f'{"Endpoint":<22} {"Labelled":>9} {"Toxic":>7} {"Safe":>7} {"% Toxic":>8} {"SPW":>6}')
print('-' * 62)
for _, r in bal_df.iterrows():
    print(f'{r["endpoint"]:<22} {r["n_total"]:>9,} {r["n_toxic"]:>7,} '
          f'{r["n_safe"]:>7,} {r["pct_toxic"]:>7.1f}% {r["scale_pos_weight"]:>5.1f}x')

with open('scale_pos_weights.json', 'w') as f:
    json.dump(spw_dict, f, indent=2)
print('\nSaved → scale_pos_weights.json')


Endpoint                Labelled   Toxic    Safe  % Toxic    SPW
--------------------------------------------------------------
NR-AR                      7,258     308   6,950     4.2%  22.6x
NR-AR-LBD                  6,751     237   6,514     3.5%  27.5x
NR-AhR                     6,542     768   5,774    11.7%   7.5x
NR-Aromatase               5,815     300   5,515     5.2%  18.4x
NR-ER                      6,186     791   5,395    12.8%   6.8x
NR-ER-LBD                  6,948     349   6,599     5.0%  18.9x
NR-PPAR-gamma              6,443     186   6,257     2.9%  33.6x
SR-ARE                     5,825     942   4,883    16.2%   5.2x
SR-ATAD5                   7,065     264   6,801     3.7%  25.8x
SR-HSE                     6,460     372   6,088     5.8%  16.4x
SR-MMP                     5,804     918   4,886    15.8%   5.3x
SR-p53                     6,767     423   6,344     6.3%  15.0x

Saved → scale_pos_weights.json


---
# Feature Engineering

## 1st Layer Morgan ECFP4 with chirality (radius=2, 2048 bits)



In [8]:
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=True
)
morgan_df = pd.DataFrame(
    [list(morgan_gen.GetFingerprint(mol)) for mol in mols],
    columns=[f'morgan_{i}' for i in range(2048)]
)
print(f'Layer 1A Morgan ECFP4 (chirality) : {morgan_df.shape}')
print(f'Avg bits ON : {morgan_df.sum(axis=1).mean():.1f} / 2048')
print(f'Sparsity    : {100*(1 - morgan_df.mean().mean()):.1f}% zeros')


Layer 1A Morgan ECFP4 (chirality) : (7823, 2048)
Avg bits ON : 29.9 / 2048
Sparsity    : 98.5% zeros


## 2nd Layer — Atom Pair Fingerprints (1024 bits)




In [9]:
from rdkit.Chem import rdMolDescriptors

def get_atom_pair_fp(mol):
    fp = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True
    )
    return list(fp)

ap_df = pd.DataFrame(
    [get_atom_pair_fp(mol) for mol in mols],
    columns=[f'ap_{i}' for i in range(1024)]
)
print(f'Layer 1B Atom Pair fingerprints : {ap_df.shape}')
print(f'Avg bits ON : {ap_df.sum(axis=1).mean():.1f} / 1024')


[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:52] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:53] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:53] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:53] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:53] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:53] DEPRECATION WARNING: please use AtomPairGenerator
[01:05:53] DEPRECATION W

Layer 1B Atom Pair fingerprints : (7823, 1024)
Avg bits ON : 132.8 / 1024


## 3rd Layer — Topological Torsion Fingerprints (1024 bits)




In [10]:
def get_torsion_fp(mol):
    fp = rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True
    )
    return list(fp)

tt_df = pd.DataFrame(
    [get_torsion_fp(mol) for mol in mols],
    columns=[f'tt_{i}' for i in range(1024)]
)
print(f'Layer 1C Topological Torsion : {tt_df.shape}')
print(f'Avg bits ON : {tt_df.sum(axis=1).mean():.1f} / 1024')


[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06:06] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[01:06

Layer 1C Topological Torsion : (7823, 1024)
Avg bits ON : 28.1 / 1024


## 4th Layer — MACCS Structural Keys (167 bits)

In [11]:
maccs_df = pd.DataFrame(
    [list(MACCSkeys.GenMACCSKeys(mol)) for mol in mols],
    columns=[f'maccs_{i}' for i in range(167)]
)
print(f'Shape     : {maccs_df.shape}')
print(f'NaN count : {maccs_df.isna().sum().sum()} (always 0)')


Shape     : (7823, 167)
NaN count : 0 (always 0)


## 5th Layer — Core RDKit Descriptors (10 properties)

The original 10 physicochemical descriptors: MW, logP, QED, TPSA, HBD, HBA,
RotBonds, ArRings, FracCSP3, HeavyAtoms.


In [12]:
DESC_COLUMNS = [
    'MW','logP','QED','TPSA','HBD','HBA',
    'RotBonds','ArRings','FracCSP3','HeavyAtoms'
]

def compute_descriptors(mol):
    try:
        return {
            'MW'        : Descriptors.MolWt(mol),
            'logP'      : Descriptors.MolLogP(mol),
            'QED'       : QED.qed(mol),
            'TPSA'      : CalcTPSA(mol),
            'HBD'       : CalcNumHBD(mol),
            'HBA'       : CalcNumHBA(mol),
            'RotBonds'  : CalcNumRotatableBonds(mol),
            'ArRings'   : CalcNumAromaticRings(mol),
            'FracCSP3'  : CalcFractionCSP3(mol),
            'HeavyAtoms': mol.GetNumHeavyAtoms(),
        }
    except Exception:
        return {col: np.nan for col in DESC_COLUMNS}

desc_df = pd.DataFrame([compute_descriptors(mol) for mol in mols])
print(f'Shape     : {desc_df.shape}')
print(f'NaN count : {desc_df.isna().sum().sum()}')
print(desc_df[['MW','logP','QED','TPSA']].describe().round(2))


[01:06:37] WARNING: not removing hydrogen atom without neighbors


Shape     : (7823, 10)
NaN count : 0
            MW     logP      QED     TPSA
count  7823.00  7823.00  7823.00  7823.00
mean    276.14     2.37     0.55    59.47
std     164.73     2.30     0.19    57.79
min       9.01   -17.41     0.01     0.00
25%     165.24     1.15     0.43    26.30
50%     240.30     2.37     0.55    46.53
75%     343.04     3.65     0.69    77.03
max    1877.66    22.61     0.94   777.98


## 6th Layer — Extended RDKit Descriptors (~200 properties) 



RDKit has ~200 2D descriptors covering:
- **VSA descriptors** (MOE-style surface area bins by logP/charge/H-bond)
- **Estate indices** (electrochemical atom environments)
- **Ring system descriptors** (bridgehead atoms, spiro atoms)
- **Connectivity indices** (chi0, chi1... molecular graph topology)
- **Kappa shape indices** (branching and flexibility)
- **Fragment counts** (number of each functional group type)



In [13]:

_all_desc = Descriptors.descList  

_exclude = set(DESC_COLUMNS) | {'MolLogP', 'MolWt'}

def compute_extended_descriptors(mol):
    result = {}
    for name, func in _all_desc:
        if name in _exclude:
            continue
        try:
            val = func(mol)
            
            if val is not None and np.isfinite(float(val)):
                result[name] = float(val)
            else:
                result[name] = np.nan
        except Exception:
            result[name] = np.nan
    return result

print('Computing extended descriptors for all molecules...')
ext_rows = [compute_extended_descriptors(mol) for mol in mols]
ext_df   = pd.DataFrame(ext_rows)
ext_df   = ext_df.replace([np.inf, -np.inf], np.nan)

nan_frac  = ext_df.isna().mean()
keep_cols = nan_frac[nan_frac < 0.5].index
ext_df    = ext_df[keep_cols]

print(f'Shape after NaN filter  : {ext_df.shape}')
print(f'NaN count remaining     : {ext_df.isna().sum().sum()}')
print(f'Descriptors included    : {list(ext_df.columns[:10])} ...')


Computing extended descriptors for all molecules...


[01:08:18] WARNING: not removing hydrogen atom without neighbors


Shape after NaN filter  : (7823, 214)
NaN count remaining     : 1538
Descriptors included    : ['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons'] ...


## 7th Layer — NR-ER SMARTS Fragment Counts 

These 8 SMARTS patterns encode the pharmacophore features that define
estrogen receptor binding — phenols, hydroxyl groups, steroid-like rings.

All 2D. No conformer. Direct biological signal for NR-ER and NR-ER-LBD.

|


In [14]:
from rdkit.Chem import MolFromSmarts

# SMARTS patterns — biologically motivated for NR-ER / NR-ER-LBD
NR_ER_SMARTS = {
    'smarts_phenol'          : MolFromSmarts('c1ccc(O)cc1'),          # phenol
    'smarts_aliphatic_OH'    : MolFromSmarts('[C;!a][OH]'),            # aliphatic hydroxyl
    'smarts_aromatic_ring'   : MolFromSmarts('c1ccccc1'),              # benzene ring
    'smarts_fused_6_6'       : MolFromSmarts('c1ccc2ccccc2c1'),        # fused 6-6 aromatic
    'smarts_steroid_ABC'     : MolFromSmarts('[C]1CC[C]2CC[C]3CC[C@@H](CC3)[C@@H]2[C@@H]1'), # steroid proxy
    'smarts_halogenated_aryl': MolFromSmarts('c1ccc([F,Cl,Br,I])cc1'), # halogenated phenyl
    'smarts_amine'           : MolFromSmarts('[NX3;H2,H1;!$(NC=O)]'),  # primary/secondary amine
    'smarts_carbonyl'        : MolFromSmarts('[C;!R]=O'),              # non-ring carbonyl
}

def compute_smarts_features(mol):
    result = {}
    for name, patt in NR_ER_SMARTS.items():
        try:
            if patt is None:
                result[name] = 0
            else:
                matches = mol.GetSubstructMatches(patt)
                result[name] = len(matches)
        except Exception:
            result[name] = 0
    return result

smarts_df = pd.DataFrame([compute_smarts_features(mol) for mol in mols])
print(f'Layer 3C SMARTS fragment counts : {smarts_df.shape}')
print(smarts_df.describe().round(2))


Layer 3C SMARTS fragment counts : (7823, 8)
       smarts_phenol  smarts_aliphatic_OH  smarts_aromatic_ring  \
count        7823.00              7823.00               7823.00   
mean            0.36                 0.49                  0.84   
std             0.90                 1.23                  0.93   
min             0.00                 0.00                  0.00   
25%             0.00                 0.00                  0.00   
50%             0.00                 0.00                  1.00   
75%             0.00                 1.00                  1.00   
max            30.00                24.00                 10.00   

       smarts_fused_6_6  smarts_steroid_ABC  smarts_halogenated_aryl  \
count           7823.00              7823.0                  7823.00   
mean               0.04                 0.0                     0.24   
std                0.29                 0.0                     0.71   
min                0.00                 0.0                     

## 8th Layer — Engineered Features 

In [15]:
eng = pd.DataFrame()

eng['lip_pass']       = (
    (desc_df['MW'] < 500) & (desc_df['logP'] < 5) &
    (desc_df['HBD'] <= 5) & (desc_df['HBA'] <= 10)
).astype(int)

eng['lip_violations'] = (
    (desc_df['MW'] >= 500).astype(int) +
    (desc_df['logP'] >= 5).astype(int) +
    (desc_df['HBD'] > 5).astype(int)  +
    (desc_df['HBA'] > 10).astype(int)
)

eng['logP_x_MW']   = desc_df['logP'] * desc_df['MW']
eng['TPSA_per_MW'] = desc_df['TPSA'] / (desc_df['MW'] + 1)
eng['HBD_HBA_sum'] = desc_df['HBD'] + desc_df['HBA']
eng['logMW']       = np.log1p(desc_df['MW'])

eng = eng.replace([np.inf, -np.inf], np.nan)

print(f'Shape     : {eng.shape}')
print(f'Features  : {list(eng.columns)}')
print(f'NaN count : {eng.isna().sum().sum()}')


Shape     : (7823, 6)
Features  : ['lip_pass', 'lip_violations', 'logP_x_MW', 'TPSA_per_MW', 'HBD_HBA_sum', 'logMW']
NaN count : 0


## 9th Layer — ZINC-style Drug-likeness Properties

In [16]:
try:
    from rdkit import RDConfig
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
    import sascorer
    HAS_SAS = True
    print('SAS scorer loaded ✓')
except Exception:
    HAS_SAS = False
    print('SAS scorer not found — using structural estimate')

def compute_zinc_features(mol):
    try:
        mw   = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        qed  = QED.qed(mol)
        if HAS_SAS:
            try:    sas = sascorer.calculateScore(mol)
            except: sas = np.nan
        else:
            sas = min(max(
                1.0 + 0.25*CalcNumRings(mol)
                    + 0.04*mol.GetNumHeavyAtoms()
                    + 0.5*CalcNumAtomStereoCenters(mol)
                    + 0.8*CalcNumSpiroAtoms(mol),
                1.0), 10.0)
        return {
            'zinc_SAS'     : sas,
            'drug_like'    : int(150<mw<500 and -0.4<logp<5.6),
            'fragment_like': int(mw<300 and logp<3),
            'lead_like'    : int(250<mw<350 and -1<logp<3.5),
        }
    except Exception:
        return {k: np.nan for k in
                ['zinc_SAS','drug_like','fragment_like','lead_like']}

zinc_df = pd.DataFrame([compute_zinc_features(mol) for mol in mols])
zinc_df = zinc_df.replace([np.inf, -np.inf], np.nan)

print(f'Shape     : {zinc_df.shape}')
print(f'NaN count : {zinc_df.isna().sum().sum()}')
n_drug  = zinc_df['drug_like'].sum()
print(f'Drug-like : {int(n_drug):,}/{len(zinc_df):,} ({100*n_drug/len(zinc_df):.1f}%)')


SAS scorer loaded ✓


[01:11:25] WARNING: not removing hydrogen atom without neighbors


Shape     : (7823, 4)
NaN count : 0
Drug-like : 5,140/7,823 (65.7%)


## 10th Layer — Chirality + Formal Charge




In [17]:
def compute_chirality_charge(mol):
    try:
        atoms   = list(mol.GetAtoms())
        charges = [a.GetFormalCharge() for a in atoms]
        n_cw    = sum(1 for a in atoms if int(a.GetChiralTag()) == 1)
        n_ccw   = sum(1 for a in atoms if int(a.GetChiralTag()) == 2)
        return {
            'chiral_n_R'      : n_cw,
            'chiral_n_S'      : n_ccw,
            'chiral_total'    : n_cw + n_ccw,
            'has_chirality'   : int((n_cw + n_ccw) > 0),
            'chiral_RS_ratio' : n_cw / (n_ccw + 1e-6),
            'total_charge'    : sum(charges),
            'n_pos_atoms'     : sum(1 for c in charges if c > 0),
            'n_neg_atoms'     : sum(1 for c in charges if c < 0),
            'max_atom_charge' : max(charges) if charges else 0,
            'charge_range'    : max(charges) - min(charges) if charges else 0,
        }
    except Exception:
        return {k: np.nan for k in [
            'chiral_n_R','chiral_n_S','chiral_total','has_chirality',
            'chiral_RS_ratio','total_charge','n_pos_atoms','n_neg_atoms',
            'max_atom_charge','charge_range'
        ]}

chiral_df = pd.DataFrame([compute_chirality_charge(mol) for mol in mols])
print(f'Shape     : {chiral_df.shape}')
print(f'NaN count : {chiral_df.isna().sum().sum()}')
print(f'Molecules with chirality : {int(chiral_df["has_chirality"].sum()):,} '
      f'({100*chiral_df["has_chirality"].mean():.1f}%)')


Shape     : (7823, 10)
NaN count : 0
Molecules with chirality : 1,321 (16.9%)


---
## Combining All Layers + KNN Imputation + VarianceThreshold



## 11th Layer -  AUTOCORR3D + WHIM 3D Descriptors


In [18]:
mols_3d = []
for i, mol in enumerate(mols):
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    try:
        AllChem.MMFFOptimizeMolecule(mol)
        mols_3d.append(mol)
    except Exception:
        mols_3d.append(None) 

print(f'Generated 3D conformers for {len(mols_3d)} molecules. ({mols_3d.count(None)} failed)')

[01:11:59] UFFTYPER: Unrecognized charge state for atom: 0
[01:11:59] UFFTYPER: Unrecognized atom type: Zn+2 (0)
[01:12:03] UFFTYPER: Unrecognized atom type: Ca+2 (0)
[01:12:04] UFFTYPER: Unrecognized charge state for atom: 6
[01:12:12] UFFTYPER: Unrecognized charge state for atom: 7
[01:12:12] UFFTYPER: Unrecognized atom type: Cu6+1 (0)
[01:12:17] UFFTYPER: Warning: hybridization set to SP3 for atom 0
[01:12:17] UFFTYPER: Unrecognized charge state for atom: 0
[01:12:17] UFFTYPER: Unrecognized charge state for atom: 4
[01:12:18] UFFTYPER: Unrecognized charge state for atom: 4
[01:13:02] UFFTYPER: Unrecognized charge state for atom: 8
[01:13:05] UFFTYPER: Warning: hybridization set to SP for atom 0
[01:13:05] UFFTYPER: Unrecognized charge state for atom: 0
[01:13:09] UFFTYPER: Unrecognized atom type: Cr3+3 (1)
[01:13:09] UFFTYPER: Unrecognized atom type: Cr3+3 (5)
[01:13:16] UFFTYPER: Unrecognized atom type: Cr3+3 (4)
[01:13:16] UFFTYPER: Unrecognized atom type: Cr3+3 (4)
[01:13:16] UFF

Generated 3D conformers for 7823 molecules. (22 failed)


In [19]:
from rdkit.Chem import rdMolDescriptors

def compute_autocorr3d(mol_3d):
    """
    Returns 80 autocorrelation values.
    Each value = correlation of an atomic property at a given distance.
    Handles failed conformers by returning NaN row.
    """
    if mol_3d is None:
        return {f'autocorr3d_{i}': np.nan for i in range(80)}
    try:
        ac = rdMolDescriptors.CalcAUTOCORR3D(mol_3d)
        return {f'autocorr3d_{i}': v for i, v in enumerate(ac)}
    except Exception:
        return {f'autocorr3d_{i}': np.nan for i in range(80)}

autocorr3d_df = pd.DataFrame([compute_autocorr3d(m) for m in mols_3d])
print(f'AUTOCORR3D shape : {autocorr3d_df.shape}')
print(f'NaN count        : {autocorr3d_df.isna().sum().sum()}')


def compute_whim(mol_3d):
    """
    Returns 114 WHIM descriptor values.
    Handles failed conformers by returning NaN row.
    """
    if mol_3d is None:
        return {f'whim_{i}': np.nan for i in range(114)}
    try:
        whim = rdMolDescriptors.CalcWHIM(mol_3d)
        return {f'whim_{i}': v for i, v in enumerate(whim)}
    except Exception:
        return {f'whim_{i}': np.nan for i in range(114)}

whim_df = pd.DataFrame([compute_whim(m) for m in mols_3d])
print(f'WHIM shape before VT : {whim_df.shape}')
print(f'NaN count            : {whim_df.isna().sum().sum()}')

whim_vals = whim_df.values.copy()
col_medians = np.nanmedian(whim_vals, axis=0)
nan_mask = np.isnan(whim_vals)
whim_vals[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])

vt_whim = VarianceThreshold(threshold=0.01)
whim_filtered = vt_whim.fit_transform(whim_vals)
kept_whim_cols = [whim_df.columns[i] for i in range(114) if vt_whim.get_support()[i]]
whim_filtered_df = pd.DataFrame(whim_filtered, columns=kept_whim_cols)
print(f'WHIM shape after VT  : {whim_filtered_df.shape}')
print(f'Removed {114 - whim_filtered_df.shape[1]} near-zero WHIM features')

autocorr_vals = autocorr3d_df.values.copy()
col_med_ac = np.nanmedian(autocorr_vals, axis=0)
nan_mask_ac = np.isnan(autocorr_vals)
autocorr_vals[nan_mask_ac] = np.take(col_med_ac, np.where(nan_mask_ac)[1])
autocorr3d_clean_df = pd.DataFrame(autocorr_vals, columns=autocorr3d_df.columns)

print(f'\nFinal 3D feature counts:')
print(f'  AUTOCORR3D : {autocorr3d_clean_df.shape[1]} features')
print(f'  WHIM       : {whim_filtered_df.shape[1]} features (after VT)')
print(f'  Total 3D   : {autocorr3d_clean_df.shape[1] + whim_filtered_df.shape[1]} features')

[01:30:17] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 233 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[01:30:17] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 233 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[01:30:17] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 233 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[01:30:17] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 233 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\AUTOCORR3D.cpp
Failed Expression: mo

AUTOCORR3D shape : (7823, 80)
NaN count        : 3040


[01:30:19] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[01:30:19] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[01:30:20] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 1
****

[01:30:20] 

****
Pre-condition Violation
molecule has no conformers
Violation occurred on line 382 in file C:\rdkit\build\temp.win-amd64-cpython-313\Release\rdkit\Code\GraphMol\Descriptors\WHIM.cpp
Failed Expression: mol.getNumConformers() >= 

WHIM shape before VT : (7823, 114)
NaN count            : 3279
WHIM shape after VT  : (7823, 106)
Removed 8 near-zero WHIM features

Final 3D feature counts:
  AUTOCORR3D : 80 features
  WHIM       : 106 features (after VT)
  Total 3D   : 186 features


In [20]:

# FINAL AUGMENTATION OF LAYERS


# --- Binary fingerprint layers (VarianceThreshold applied) ---
fp_matrix = pd.concat([
    morgan_df.reset_index(drop=True),   # Layer 1A: 2048 bits ECFP4+chiral
    ap_df.reset_index(drop=True),       # Layer 1B: 1024 bits Atom Pair  (NEW)
    tt_df.reset_index(drop=True),       # Layer 1C: 1024 bits Topo Torsion (NEW)
    maccs_df.reset_index(drop=True),    # Layer 2:   167 bits MACCS
], axis=1)

print(f'Total fingerprint bits before VarianceThreshold : {fp_matrix.shape[1]}')

# VarianceThreshold: remove bits ON in <0.1% or >99.9% of molecules
# threshold = p*(1-p) where p=0.001  →  threshold = 0.000999
vt = VarianceThreshold(threshold=0.001 * (1 - 0.001))
fp_filtered = vt.fit_transform(fp_matrix.values)

kept_fp_cols = [fp_matrix.columns[i] for i in range(fp_matrix.shape[1])
                if vt.get_support()[i]]
fp_filtered_df = pd.DataFrame(fp_filtered, columns=kept_fp_cols)

print(f'Total fingerprint bits after  VarianceThreshold : {fp_filtered_df.shape[1]}')
print(f'Removed {fp_matrix.shape[1] - fp_filtered_df.shape[1]} near-zero bits')

# --- Continuous layers (KNN imputation applied) ---
continuous_df = pd.concat([
    desc_df.reset_index(drop=True),         
    ext_df.reset_index(drop=True),          
    smarts_df.reset_index(drop=True),       
    eng.reset_index(drop=True),             
    zinc_df.reset_index(drop=True),         
    chiral_df.reset_index(drop=True),       
    autocorr3d_clean_df.reset_index(drop=True),  
    whim_filtered_df.reset_index(drop=True),     
], axis=1)

print(f'\nContinuous features : {continuous_df.shape[1]} cols')
n_nan = continuous_df.isna().sum().sum()
print(f'NaN in continuous   : {n_nan}')

if n_nan > 0:
    knn = KNNImputer(n_neighbors=5, weights='distance')
    cont_imputed = pd.DataFrame(
        knn.fit_transform(continuous_df.values),
        columns=continuous_df.columns
    )
    print(f'NaN after KNN imputation : {cont_imputed.isna().sum().sum()} (must be 0)')
else:
    cont_imputed = continuous_df.copy()
    print('No NaN — KNN skipped')


cont_imputed = cont_imputed.replace([np.inf, -np.inf], np.nan)

cont_imputed = cont_imputed.fillna(cont_imputed.median())

cont_imputed = cont_imputed.clip(lower=-1e6, upper=1e6)


feature_matrix = pd.concat([
    fp_filtered_df.reset_index(drop=True),
    cont_imputed.reset_index(drop=True),
], axis=1)

feat_cols    = feature_matrix.columns.tolist()
label_matrix = df[toxicity_labels].reset_index(drop=True)

print(f'\n{"="*55}')
print(f'FINAL FEATURE MATRIX')
print(f'{"="*55}')
print(f'  Layer 1A Morgan ECFP4+chiral  : 2048 bits (before VT)')
print(f'  Layer 1B Atom Pair            : 1024 bits (NEW, before VT)')
print(f'  Layer 1C Topological Torsion  : 1024 bits (NEW, before VT)')
print(f'  Layer 2  MACCS                :  167 bits (before VT)')
print(f'  After VarianceThreshold       : {fp_filtered_df.shape[1]} bits kept')
print(f'  Layer 3A Core descriptors     : {desc_df.shape[1]} cols')
print(f'  Layer 3B Extended descriptors : {ext_df.shape[1]} cols')
print(f'  Layer 3C SMARTS NR-ER         : {smarts_df.shape[1]} cols (NEW)')
print(f'  Layer 4  Engineered           : {eng.shape[1]} cols')
print(f'  Layer 5  ZINC                 : {zinc_df.shape[1]} cols')
print(f'  +10      Chirality/charge     : 10 cols')
print(f'  Layer 6A AUTOCORR3D           : {autocorr3d_clean_df.shape[1]} cols (NEW)')
print(f'  Layer 6B WHIM (post-VT)       : {whim_filtered_df.shape[1]} cols (NEW)')
print(f'  TOTAL                         : {len(feat_cols)} cols')
print(f'{"="*55}')

# Save
final_df = pd.concat([
    df[['smiles']].reset_index(drop=True),
    feature_matrix,
    label_matrix,
], axis=1)
final_df.to_csv('features_raw.csv', index=False)
with open('feature_cols.json', 'w') as f:
    json.dump(feat_cols, f)
print('\nSaved → features_raw.csv')
print('Saved → feature_cols.json')


Total fingerprint bits before VarianceThreshold : 4263
Total fingerprint bits after  VarianceThreshold : 3971
Removed 292 near-zero bits

Continuous features : 438 cols
NaN in continuous   : 1538
NaN after KNN imputation : 0 (must be 0)

FINAL FEATURE MATRIX
  Layer 1A Morgan ECFP4+chiral  : 2048 bits (before VT)
  Layer 1B Atom Pair            : 1024 bits (NEW, before VT)
  Layer 1C Topological Torsion  : 1024 bits (NEW, before VT)
  Layer 2  MACCS                :  167 bits (before VT)
  After VarianceThreshold       : 3971 bits kept
  Layer 3A Core descriptors     : 10 cols
  Layer 3B Extended descriptors : 214 cols
  Layer 3C SMARTS NR-ER         : 8 cols (NEW)
  Layer 4  Engineered           : 6 cols
  Layer 5  ZINC                 : 4 cols
  +10      Chirality/charge     : 10 cols
  Layer 6A AUTOCORR3D           : 80 cols (NEW)
  Layer 6B WHIM (post-VT)       : 106 cols (NEW)
  TOTAL                         : 4409 cols

Saved → features_raw.csv
Saved → feature_cols.json


---
## Testing Model 1 — XGBoost Baseline with 5-fold Cross-Validation (Baseline Evaluation)


In [23]:
# ══════════════════════════════════════════════════════════════════
# STAGE 1 — XGBoost  |  High Recall + High PR-AUC edition
# ══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import pickle, os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    precision_recall_curve
)
from xgboost import XGBClassifier

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
    HAS_SMOTE = True
    print("SMOTE / BorderlineSMOTE / ADASYN available ✓")
except ImportError:
    HAS_SMOTE = False
    print("pip install imbalanced-learn")


# ════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════

def tune_threshold_recall_precision(y_true, y_prob,
                                     min_recall=0.65,
                                     min_precision=0.35):
    """
    Find highest-precision threshold that still achieves min_recall.
    Falls back to best-F1 threshold if constraint can't be met.
    """
    prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_prob)
    mask = (rec_arr[:-1] >= min_recall) & (prec_arr[:-1] >= min_precision)
    if mask.any():
        best_idx = np.argmax(prec_arr[:-1][mask])
        return float(thresholds[mask][best_idx])
    else:
        f1s = np.where(
            (prec_arr + rec_arr) == 0, 0,
            2 * prec_arr * rec_arr / (prec_arr + rec_arr)
        )
        return float(thresholds[np.argmax(f1s[:-1])])


def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return dict(
        roc_auc   = round(roc_auc_score(y_true, y_prob),           4),
        pr_auc    = round(average_precision_score(y_true, y_prob),  4),
        recall    = round(recall_score(y_true, y_pred,    zero_division=0), 4),
        precision = round(precision_score(y_true, y_pred, zero_division=0), 4),
        f1        = round(f1_score(y_true, y_pred,        zero_division=0), 4),
        threshold = round(threshold, 4),
    )


def smart_oversample(X_tr, y_tr, n_tox_fold, random_state=42):
    """
    Try ADASYN → BorderlineSMOTE → SMOTE in order.
    ADASYN focuses on hardest-to-learn borderline toxics.
    """
    if not HAS_SMOTE or n_tox_fold < 6:
        return X_tr, y_tr

    n_safe_fold  = (y_tr == 0).sum()
    target_ratio = min(0.45, (n_tox_fold * 3) / n_safe_fold)
    k            = min(5, n_tox_fold - 1)

    for Sampler, kwargs in [
        (ADASYN,          dict(sampling_strategy=target_ratio,
                               n_neighbors=k, random_state=random_state)),
        (BorderlineSMOTE, dict(sampling_strategy=target_ratio,
                               k_neighbors=k, random_state=random_state)),
        (SMOTE,           dict(sampling_strategy=target_ratio,
                               k_neighbors=k, random_state=random_state)),
    ]:
        try:
            X_res, y_res = Sampler(**kwargs).fit_resample(X_tr, y_tr)
            return X_res, y_res
        except Exception:
            continue

    return X_tr, y_tr


# ════════════════════════════════════════════════════════
# DATA PREP
# ════════════════════════════════════════════════════════
X_array = feature_matrix.values.astype(np.float32)
np.nan_to_num(X_array, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
print(f"X_array shape : {X_array.shape}")
print(f"inf count     : {np.isinf(X_array).sum()}  (must be 0)")
print(f"nan count     : {np.isnan(X_array).sum()}  (must be 0)")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── cache structures ──────────────────────────────────────────────
xgb_fold_preds  = {}
xgb_fold_aucs   = {}
xgb_fold_models = {}
xgb_models      = {}
ep_data         = {}
oof_probs       = {}
oof_labels      = {}
oof_thresholds  = {}
all_results     = []


# ════════════════════════════════════════════════════════
# XGBOOST PARAMS
# ════════════════════════════════════════════════════════
def make_xgb(spw, n_toxic):
    aggressive_spw = min(60, spw * 2.0)
    return XGBClassifier(
        n_estimators      = 500,
        max_depth         = 6,
        learning_rate     = 0.03,
        subsample         = 0.85,
        colsample_bytree  = 0.75,
        colsample_bylevel = 0.75,
        min_child_weight  = 1,            # ← allows splits on tiny minority groups
        scale_pos_weight  = aggressive_spw,
        gamma             = 0,
        max_delta_step    = 1,
        reg_alpha         = 0.1,
        reg_lambda        = 1.0,
        grow_policy       = 'lossguide',  # ← splits where loss drops most
        max_leaves        = 63,           # ← pairs with lossguide
        eval_metric       = "aucpr",
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = 0,
    )


# ════════════════════════════════════════════════════════
# HEADER
# ════════════════════════════════════════════════════════
col = (f"{'Endpoint':<24}{'N':>7}{'Toxic':>7}{'SPW':>6}  "
       f"{'ROC-AUC':>8}{'PR-AUC':>8}{'Recall':>8}"
       f"{'Precis':>8}{'F1':>8}{'Thr':>6}")
print(f"\nXGBOOST — 5-fold CV  |  ADASYN/BorderlineSMOTE + aggressive SPW")
print(f"Threshold policy: max precision s.t. recall ≥ 0.65 & precision ≥ 0.35")
print(col)
print("─" * len(col))


# ════════════════════════════════════════════════════════
# MAIN LOOP
# ════════════════════════════════════════════════════════
for ep in toxicity_labels:
    mask    = label_matrix[ep].notna()
    X_ep    = X_array[mask]
    y_ep    = label_matrix[ep][mask].values.astype(int)
    n_toxic = int((y_ep == 1).sum())
    n_safe  = int((y_ep == 0).sum())

    if n_toxic < 10:
        print(f"{ep:<24} SKIPPED  ({n_toxic} toxic samples)")
        continue

    spw = round(n_safe / n_toxic, 1)
    ep_data[ep] = (mask, X_ep, y_ep, spw)

    fold_preds = np.zeros(len(y_ep))
    fold_aucs  = []
    fold_mods  = []

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_ep, y_ep)):
        X_tr, y_tr   = X_ep[tr_idx], y_ep[tr_idx]
        X_val, y_val = X_ep[val_idx], y_ep[val_idx]

        n_tox_fold = int((y_tr == 1).sum())
        X_tr, y_tr = smart_oversample(X_tr, y_tr, n_tox_fold, random_state=42)

        m = make_xgb(spw, n_toxic)
        m.fit(X_tr, y_tr)

        probs = m.predict_proba(X_val)[:, 1]
        fold_preds[val_idx] = probs
        fold_aucs.append(roc_auc_score(y_val, probs))
        fold_mods.append(m)

    # cache
    xgb_fold_preds[ep]  = fold_preds
    xgb_fold_aucs[ep]   = fold_aucs
    xgb_fold_models[ep] = fold_mods
    oof_probs[ep]        = fold_preds
    oof_labels[ep]       = y_ep

    # threshold: target recall ≥ 0.65, precision ≥ 0.35
    best_thr = tune_threshold_recall_precision(
        y_ep, fold_preds,
        min_recall=0.65,
        min_precision=0.35
    )
    oof_thresholds[ep] = best_thr

    m_dict = compute_metrics(y_ep, fold_preds, best_thr)
    all_results.append({"endpoint": ep, "n_rows": len(y_ep),
                        "n_toxic": n_toxic, "spw": spw, **m_dict})

    print(f"{ep:<24}{len(y_ep):>7,}{n_toxic:>7,}{spw:>5.1f}x  "
          f"{m_dict['roc_auc']:>8.4f}{m_dict['pr_auc']:>8.4f}"
          f"{m_dict['recall']:>8.4f}{m_dict['precision']:>8.4f}"
          f"{m_dict['f1']:>8.4f}{best_thr:>6.2f}")

    # final model on full endpoint data
    final_xgb = make_xgb(spw, n_toxic)
    final_xgb.fit(X_ep, y_ep)
    xgb_models[ep] = final_xgb


# ════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════
xgb_df = pd.DataFrame(all_results)
numeric_cols = ["roc_auc", "pr_auc", "recall", "precision", "f1"]

print("\n" + "═" * len(col))
print("MACRO AVERAGE  (unweighted across endpoints)")
print("═" * len(col))
for c in numeric_cols:
    print(f"  {c:<12}  mean={xgb_df[c].mean():.4f}   std={xgb_df[c].std():.4f}")

print("\nWEIGHTED AVERAGE  (weighted by n_toxic)")
w = xgb_df["n_toxic"].values
for c in numeric_cols:
    print(f"  {c:<12}  weighted_mean={np.average(xgb_df[c], weights=w):.4f}")

struggling = xgb_df[(xgb_df["recall"] < 0.60) | (xgb_df["pr_auc"] < 0.50)]
if not struggling.empty:
    print(f"\n⚠  Endpoints still below target (recall<0.60 or PR-AUC<0.50):")
    print(struggling[["endpoint", "n_toxic", "spw", "pr_auc",
                       "recall", "precision"]].to_string(index=False))

print("\nFull results:")
print(xgb_df[["endpoint","n_rows","n_toxic","spw"] +
             numeric_cols + ["threshold"]].to_string(index=False))

# save
os.makedirs("models", exist_ok=True)
with open("models/xgb_models.pkl",      "wb") as f: pickle.dump(xgb_models,     f)
with open("models/oof_thresholds.pkl",  "wb") as f: pickle.dump(oof_thresholds,  f)
with open("models/oof_probs.pkl",       "wb") as f: pickle.dump(oof_probs,       f)
print("\nSaved → models/xgb_models.pkl  |  oof_thresholds.pkl  |  oof_probs.pkl")
print(f"Cached: {len(xgb_fold_preds)} endpoints × 5 folds in memory (ensemble ready)")

SMOTE / BorderlineSMOTE / ADASYN available ✓
X_array shape : (7823, 4409)
inf count     : 0  (must be 0)
nan count     : 0  (must be 0)

XGBOOST — 5-fold CV  |  ADASYN/BorderlineSMOTE + aggressive SPW
Threshold policy: max precision s.t. recall ≥ 0.65 & precision ≥ 0.35
Endpoint                      N  Toxic   SPW   ROC-AUC  PR-AUC  Recall  Precis      F1   Thr
────────────────────────────────────────────────────────────────────────────────────────────
NR-AR                     7,258    308 22.6x    0.7798  0.5332  0.4578  0.8758  0.6013  0.86
NR-AR-LBD                 6,751    237 27.5x    0.8538  0.6524  0.6540  0.5401  0.5916  0.13
NR-AhR                    6,542    768  7.5x    0.9082  0.6821  0.6510  0.6098  0.6297  0.59
NR-Aromatase              5,815    300 18.4x    0.8640  0.4705  0.3867  0.5395  0.4505  0.50
NR-ER                     6,186    791  6.8x    0.7322  0.4809  0.3717  0.5799  0.4530  0.71
NR-ER-LBD                 6,948    349 18.9x    0.8412  0.5569  0.4527  0.7054

In [24]:
# ══════════════════════════════════════════════════════════════════
# STAGE 1 — XGBoost  |  High Recall + PR-AUC + Correlation-Aware
# ══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import pickle, os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    precision_recall_curve
)
from xgboost import XGBClassifier

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
    HAS_SMOTE = True
    print("SMOTE / BorderlineSMOTE / ADASYN available ✓")
except ImportError:
    HAS_SMOTE = False
    print("pip install imbalanced-learn")


# ════════════════════════════════════════════════════════
# CORRELATION GROUPS  (from heatmap ≥ 0.35)
# ════════════════════════════════════════════════════════
CORRELATION_GROUPS = {
    'NR-AR'        : ['NR-AR-LBD', 'NR-ER-LBD', 'NR-ER', 'NR-Aromatase'],
    'NR-AR-LBD'    : ['NR-AR',     'NR-ER-LBD', 'NR-ER'],
    'NR-ER'        : ['NR-ER-LBD', 'NR-Aromatase', 'NR-AR', 'NR-AR-LBD'],
    'NR-ER-LBD'    : ['NR-ER',     'NR-AR',     'NR-AR-LBD'],
    'NR-Aromatase' : ['NR-ER',     'NR-AR'],
    'SR-ATAD5'     : ['SR-p53',    'SR-MMP'],
    'SR-p53'       : ['SR-ATAD5',  'SR-MMP',    'SR-ARE'],
    'SR-MMP'       : ['SR-ARE',    'SR-ATAD5',  'SR-p53'],
    'SR-ARE'       : ['SR-MMP',    'SR-p53'],
    'NR-AhR'       : [],
    'NR-PPAR-gamma': [],
    'SR-HSE'       : [],
}


# ════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════
def tune_threshold_recall_precision(y_true, y_prob,
                                     min_recall=0.65,
                                     min_precision=0.35):
    prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_prob)
    mask = (rec_arr[:-1] >= min_recall) & (prec_arr[:-1] >= min_precision)
    if mask.any():
        best_idx = np.argmax(prec_arr[:-1][mask])
        return float(thresholds[mask][best_idx])
    else:
        f1s = np.where(
            (prec_arr + rec_arr) == 0, 0,
            2 * prec_arr * rec_arr / (prec_arr + rec_arr)
        )
        return float(thresholds[np.argmax(f1s[:-1])])


def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return dict(
        roc_auc   = round(roc_auc_score(y_true, y_prob),           4),
        pr_auc    = round(average_precision_score(y_true, y_prob),  4),
        recall    = round(recall_score(y_true, y_pred,    zero_division=0), 4),
        precision = round(precision_score(y_true, y_pred, zero_division=0), 4),
        f1        = round(f1_score(y_true, y_pred,        zero_division=0), 4),
        threshold = round(threshold, 4),
    )


def smart_oversample(X_tr, y_tr, n_tox_fold, random_state=42):
    if not HAS_SMOTE or n_tox_fold < 6:
        return X_tr, y_tr
    n_safe_fold  = (y_tr == 0).sum()
    target_ratio = min(0.45, (n_tox_fold * 3) / n_safe_fold)
    k            = min(5, n_tox_fold - 1)
    for Sampler, kwargs in [
        (ADASYN,          dict(sampling_strategy=target_ratio,
                               n_neighbors=k, random_state=random_state)),
        (BorderlineSMOTE, dict(sampling_strategy=target_ratio,
                               k_neighbors=k, random_state=random_state)),
        (SMOTE,           dict(sampling_strategy=target_ratio,
                               k_neighbors=k, random_state=random_state)),
    ]:
        try:
            return Sampler(**kwargs).fit_resample(X_tr, y_tr)
        except Exception:
            continue
    return X_tr, y_tr


def make_xgb(spw, n_toxic):
    aggressive_spw = min(60, spw * 2.0)
    return XGBClassifier(
        n_estimators      = 500,
        max_depth         = 6,
        learning_rate     = 0.03,
        subsample         = 0.85,
        colsample_bytree  = 0.75,
        colsample_bylevel = 0.75,
        min_child_weight  = 1,
        scale_pos_weight  = aggressive_spw,
        gamma             = 0,
        max_delta_step    = 1,
        reg_alpha         = 0.1,
        reg_lambda        = 1.0,
        grow_policy       = 'lossguide',
        max_leaves        = 63,
        eval_metric       = "aucpr",
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = 0,
    )


def run_xgb_cv(ep, X_ep, y_ep, spw, extra_features=None):
    """
    Run 5-fold CV for one endpoint.
    extra_features: (n_ep_rows, k) array of neighbour OOF probs — appended to X.
    Returns fold_preds, fold_aucs, fold_mods.
    """
    n_toxic    = int((y_ep == 1).sum())
    fold_preds = np.zeros(len(y_ep))
    fold_aucs  = []
    fold_mods  = []

    X_use = X_ep if extra_features is None else np.hstack([X_ep, extra_features])

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_use, y_ep)):
        X_tr, y_tr   = X_use[tr_idx], y_ep[tr_idx]
        X_val, y_val = X_use[val_idx], y_ep[val_idx]

        n_tox_fold = int((y_tr == 1).sum())
        X_tr, y_tr = smart_oversample(X_tr, y_tr, n_tox_fold, random_state=42)

        m = make_xgb(spw, n_toxic)
        m.fit(X_tr, y_tr)

        probs = m.predict_proba(X_val)[:, 1]
        fold_preds[val_idx] = probs
        fold_aucs.append(roc_auc_score(y_val, probs))
        fold_mods.append(m)

    return fold_preds, fold_aucs, fold_mods


# ════════════════════════════════════════════════════════
# DATA PREP
# ════════════════════════════════════════════════════════
X_array = feature_matrix.values.astype(np.float32)
np.nan_to_num(X_array, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
print(f"X_array shape : {X_array.shape}")
print(f"inf count     : {np.isinf(X_array).sum()}  (must be 0)")
print(f"nan count     : {np.isnan(X_array).sum()}  (must be 0)")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── cache structures ──────────────────────────────────────────────
xgb_fold_preds  = {}
xgb_fold_aucs   = {}
xgb_fold_models = {}
xgb_models      = {}
ep_data         = {}
oof_probs       = {}
oof_labels      = {}
oof_thresholds  = {}
all_results     = []

# ════════════════════════════════════════════════════════
# PASS 1 — train all endpoints independently
# ════════════════════════════════════════════════════════
col = (f"{'Endpoint':<24}{'N':>7}{'Toxic':>7}{'SPW':>6}  "
       f"{'ROC-AUC':>8}{'PR-AUC':>8}{'Recall':>8}"
       f"{'Precis':>8}{'F1':>8}{'Thr':>6}{'Pass':>6}")
print("\nPASS 1 — Independent training (building neighbour OOF probs)")
print(col)
print("─" * len(col))

for ep in toxicity_labels:
    mask    = label_matrix[ep].notna()
    X_ep    = X_array[mask]
    y_ep    = label_matrix[ep][mask].values.astype(int)
    n_toxic = int((y_ep == 1).sum())
    n_safe  = int((y_ep == 0).sum())

    if n_toxic < 10:
        print(f"{ep:<24} SKIPPED  ({n_toxic} toxic samples)")
        continue

    spw = round(n_safe / n_toxic, 1)
    ep_data[ep] = (mask, X_ep, y_ep, spw)

    fold_preds, fold_aucs, fold_mods = run_xgb_cv(ep, X_ep, y_ep, spw)

    xgb_fold_preds[ep]  = fold_preds
    xgb_fold_aucs[ep]   = fold_aucs
    xgb_fold_models[ep] = fold_mods
    oof_probs[ep]        = fold_preds
    oof_labels[ep]       = y_ep

    best_thr = tune_threshold_recall_precision(y_ep, fold_preds)
    oof_thresholds[ep] = best_thr
    m_dict = compute_metrics(y_ep, fold_preds, best_thr)

    print(f"{ep:<24}{len(y_ep):>7,}{n_toxic:>7,}{spw:>5.1f}x  "
          f"{m_dict['roc_auc']:>8.4f}{m_dict['pr_auc']:>8.4f}"
          f"{m_dict['recall']:>8.4f}{m_dict['precision']:>8.4f}"
          f"{m_dict['f1']:>8.4f}{best_thr:>6.2f}{'  1':>6}")


# ════════════════════════════════════════════════════════
# PASS 2 — retrain correlated endpoints with neighbour
#          OOF probs appended as extra features
# ════════════════════════════════════════════════════════
print("\nPASS 2 — Correlation-aware retraining (neighbour OOF as features)")
print(col)
print("─" * len(col))

for ep in toxicity_labels:
    if ep not in ep_data:
        continue

    neighbours = [n for n in CORRELATION_GROUPS.get(ep, [])
                  if n in oof_probs]

    if not neighbours:
        # standalone endpoint — keep Pass 1 results as-is
        continue

    mask, X_ep, y_ep, spw = ep_data[ep]
    n_toxic = int((y_ep == 1).sum())

    # Build neighbour OOF feature matrix — aligned to this endpoint's rows
    # Each neighbour contributes one column of OOF probabilities
    neighbour_cols = []
    for nb in neighbours:
        nb_mask     = label_matrix[nb].notna()
        # full-length neighbour OOF (NaN where nb has no label)
        nb_oof_full = np.full(len(label_matrix), np.nan)
        nb_idx      = np.where(nb_mask)[0]
        nb_oof_full[nb_idx] = oof_probs[nb]

        # align to this endpoint's rows
        ep_idx      = np.where(mask)[0]
        nb_col      = nb_oof_full[ep_idx]

        # fill missing with 0.0 (molecule has no label for neighbour)
        nb_col      = np.nan_to_num(nb_col, nan=0.0)
        neighbour_cols.append(nb_col)

    extra_features = np.column_stack(neighbour_cols).astype(np.float32)

    fold_preds, fold_aucs, fold_mods = run_xgb_cv(
        ep, X_ep, y_ep, spw, extra_features=extra_features
    )

    # overwrite Pass 1 cache with Pass 2 (better) results
    xgb_fold_preds[ep]  = fold_preds
    xgb_fold_aucs[ep]   = fold_aucs
    xgb_fold_models[ep] = fold_mods
    oof_probs[ep]        = fold_preds
    oof_labels[ep]       = y_ep

    best_thr = tune_threshold_recall_precision(y_ep, fold_preds)
    oof_thresholds[ep] = best_thr
    m_dict = compute_metrics(y_ep, fold_preds, best_thr)

    all_results.append({"endpoint": ep, "n_rows": len(y_ep),
                        "n_toxic": n_toxic, "spw": spw, **m_dict})

    print(f"{ep:<24}{len(y_ep):>7,}{n_toxic:>7,}{spw:>5.1f}x  "
          f"{m_dict['roc_auc']:>8.4f}{m_dict['pr_auc']:>8.4f}"
          f"{m_dict['recall']:>8.4f}{m_dict['precision']:>8.4f}"
          f"{m_dict['f1']:>8.4f}{best_thr:>6.2f}{'  2':>6}")

    # also retrain final model on full data with neighbour features
    X_ep_aug = np.hstack([X_ep, extra_features])
    final_xgb = make_xgb(spw, n_toxic)
    final_xgb.fit(X_ep_aug, y_ep)
    xgb_models[ep] = final_xgb

# for standalone endpoints, train final model normally
for ep in toxicity_labels:
    if ep not in ep_data or ep in xgb_models:
        continue
    mask, X_ep, y_ep, spw = ep_data[ep]
    n_toxic = int((y_ep == 1).sum())
    final_xgb = make_xgb(spw, n_toxic)
    final_xgb.fit(X_ep, y_ep)
    xgb_models[ep] = final_xgb

    # add to all_results for standalone endpoints
    best_thr = oof_thresholds[ep]
    m_dict   = compute_metrics(y_ep, oof_probs[ep], best_thr)
    all_results.append({"endpoint": ep, "n_rows": len(y_ep),
                        "n_toxic": n_toxic, "spw": spw, **m_dict})


# ════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════
xgb_df = pd.DataFrame(all_results)
numeric_cols = ["roc_auc", "pr_auc", "recall", "precision", "f1"]

print("\n" + "═" * len(col))
print("FINAL SUMMARY — Pass 2 results for correlated, Pass 1 for standalone")
print("═" * len(col))
for c in numeric_cols:
    print(f"  {c:<12}  mean={xgb_df[c].mean():.4f}   std={xgb_df[c].std():.4f}")

print("\nWEIGHTED AVERAGE  (weighted by n_toxic)")
w = xgb_df["n_toxic"].values
for c in numeric_cols:
    print(f"  {c:<12}  weighted_mean={np.average(xgb_df[c], weights=w):.4f}")

struggling = xgb_df[(xgb_df["recall"] < 0.60) | (xgb_df["pr_auc"] < 0.50)]
if not struggling.empty:
    print(f"\n⚠  Still below target (recall<0.60 or PR-AUC<0.50):")
    print(struggling[["endpoint","n_toxic","spw","pr_auc",
                       "recall","precision"]].to_string(index=False))

print("\nFull results:")
print(xgb_df[["endpoint","n_rows","n_toxic","spw"] +
             numeric_cols + ["threshold"]].to_string(index=False))

# save
os.makedirs("models", exist_ok=True)
with open("models/xgb_models.pkl",      "wb") as f: pickle.dump(xgb_models,     f)
with open("models/oof_thresholds.pkl",  "wb") as f: pickle.dump(oof_thresholds,  f)
with open("models/oof_probs.pkl",       "wb") as f: pickle.dump(oof_probs,       f)
print("\nSaved → models/xgb_models.pkl  |  oof_thresholds.pkl  |  oof_probs.pkl")
print(f"Cached: {len(xgb_fold_preds)} endpoints × 5 folds in memory (ensemble ready)")

SMOTE / BorderlineSMOTE / ADASYN available ✓
X_array shape : (7823, 4409)
inf count     : 0  (must be 0)
nan count     : 0  (must be 0)

PASS 1 — Independent training (building neighbour OOF probs)
Endpoint                      N  Toxic   SPW   ROC-AUC  PR-AUC  Recall  Precis      F1   Thr  Pass
──────────────────────────────────────────────────────────────────────────────────────────────────
NR-AR                     7,258    308 22.6x    0.7798  0.5332  0.4578  0.8758  0.6013  0.86     1
NR-AR-LBD                 6,751    237 27.5x    0.8538  0.6524  0.6540  0.5401  0.5916  0.13     1
NR-AhR                    6,542    768  7.5x    0.9082  0.6821  0.6510  0.6098  0.6297  0.59     1
NR-Aromatase              5,815    300 18.4x    0.8640  0.4705  0.3867  0.5395  0.4505  0.50     1
NR-ER                     6,186    791  6.8x    0.7322  0.4809  0.3717  0.5799  0.4530  0.71     1
NR-ER-LBD                 6,948    349 18.9x    0.8412  0.5569  0.4527  0.7054  0.5515  0.68     1
NR-PPAR-ga


## Stage 2 — Cross-Endpoint Meta-Feature Construction


In [27]:
# ══════════════════════════════════════════════════════════════════
# STAGE 2 — Cross-Endpoint OOF Meta-Features
# ══════════════════════════════════════════════════════════════════

# Build full-length OOF arrays (NaN where molecule has no label)
oof_feature_rows = {}
for ep, probs in oof_probs.items():
    full_arr = np.full(len(df), np.nan)
    mask     = label_matrix[ep].notna()
    full_arr[np.where(mask)[0]] = probs
    oof_feature_rows[f'oof_{ep}'] = full_arr          # ← key named here

oof_feat_df = pd.DataFrame(oof_feature_rows, dtype=np.float32)  # ← force float32

# Column-wise mean fill
oof_feat_df = oof_feat_df.fillna(oof_feat_df.mean(axis=0))

# Sanity checks
assert oof_feat_df.isna().sum().sum() == 0, "NaNs remain after fillna!"
assert oof_feat_df.select_dtypes(include='object').empty,  "Object dtype columns found!"

print(f'Cross-endpoint meta-feature matrix : {oof_feat_df.shape}')
print(f'Columns  : {list(oof_feat_df.columns)}')
print(f'Dtypes   : {oof_feat_df.dtypes.unique()}')
print(f'NaN count: {oof_feat_df.isna().sum().sum()}  (must be 0)')
print(f'\nPer-column mean (toxicity signal per endpoint):')
print(oof_feat_df.mean(axis=0).round(3).to_string())   # ← axis=0 explicit

# Augment feature matrix
X_array_aug = np.hstack([X_array, oof_feat_df.values.astype(np.float32)])
np.nan_to_num(X_array_aug, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

print(f'\nOriginal X shape  : {X_array.shape}')
print(f'Augmented X shape : {X_array_aug.shape}  '
      f'(+{X_array_aug.shape[1] - X_array.shape[1]} meta-features)')

# Build per-endpoint augmented data — used by Stage 3 ensemble
ep_data_aug = {}
for ep, (mask, X_ep, y_ep, spw) in ep_data.items():
    ep_data_aug[ep] = (mask, X_array_aug[mask], y_ep, spw)

print(f'\nep_data_aug ready → {len(ep_data_aug)} endpoints for Stage 3 ensemble.')

Cross-endpoint meta-feature matrix : (7823, 12)
Columns  : ['oof_NR-AR', 'oof_NR-AR-LBD', 'oof_NR-AhR', 'oof_NR-Aromatase', 'oof_NR-ER', 'oof_NR-ER-LBD', 'oof_NR-PPAR-gamma', 'oof_SR-ARE', 'oof_SR-ATAD5', 'oof_SR-HSE', 'oof_SR-MMP', 'oof_SR-p53']
Dtypes   : [dtype('float32')]
NaN count: 0  (must be 0)

Per-column mean (toxicity signal per endpoint):
oof_NR-AR            0.048
oof_NR-AR-LBD        0.036
oof_NR-AhR           0.174
oof_NR-Aromatase     0.059
oof_NR-ER            0.245
oof_NR-ER-LBD        0.067
oof_NR-PPAR-gamma    0.030
oof_SR-ARE           0.269
oof_SR-ATAD5         0.045
oof_SR-HSE           0.099
oof_SR-MMP           0.205
oof_SR-p53           0.091

Original X shape  : (7823, 4409)
Augmented X shape : (7823, 4421)  (+12 meta-features)

ep_data_aug ready → 12 endpoints for Stage 3 ensemble.



## Stage 3 — Endpoint-Aware Ensemble



In [28]:
# ══════════════════════════════════════════════════════════════════
# STAGE 3 — Correlation-Aware Ensemble
# Targets: high PR-AUC, high Recall, decent Precision
# Weights optimised per endpoint on OOF (not hardcoded)
# ══════════════════════════════════════════════════════════════════
from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    recall_score, precision_score, f1_score,
    precision_recall_curve
)

# ── Correlation clusters from heatmap ────────────────────────────
CORRELATION_GROUPS = {
    'NR-AR'        : ['NR-AR-LBD', 'NR-ER-LBD', 'NR-ER', 'NR-Aromatase'],
    'NR-AR-LBD'    : ['NR-AR',     'NR-ER-LBD', 'NR-ER'],
    'NR-ER'        : ['NR-ER-LBD', 'NR-Aromatase', 'NR-AR', 'NR-AR-LBD'],
    'NR-ER-LBD'    : ['NR-ER',     'NR-AR',     'NR-AR-LBD'],
    'NR-Aromatase' : ['NR-ER',     'NR-AR'],
    'SR-ATAD5'     : ['SR-p53',    'SR-MMP'],
    'SR-p53'       : ['SR-ATAD5',  'SR-MMP',    'SR-ARE'],
    'SR-MMP'       : ['SR-ARE',    'SR-ATAD5',  'SR-p53'],
    'SR-ARE'       : ['SR-MMP',    'SR-p53'],
    'NR-AhR'       : [],
    'NR-PPAR-gamma': [],
    'SR-HSE'       : [],
}

# ── Tier assignment ───────────────────────────────────────────────
# Special  → heavily imbalanced + strong correlations → CAT + BRF
# RF-heavy → moderate imbalance, RF generalises well
# XGB-heavy→ weaker correlation signal, XGB dominates
SPECIAL_ENDPOINTS   = {'NR-AR', 'NR-AR-LBD', 'NR-ER'}
RF_STRONG_ENDPOINTS = {'NR-PPAR-gamma', 'SR-ATAD5', 'SR-HSE', 'NR-ER-LBD'}
RF_WEAK_ENDPOINTS   = {'NR-AhR', 'NR-Aromatase', 'SR-ARE', 'SR-MMP', 'SR-p53'}


def tune_threshold_recall_precision(y_true, y_prob,
                                     min_recall=0.65,
                                     min_precision=0.35):
    prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_prob)
    mask = (rec_arr[:-1] >= min_recall) & (prec_arr[:-1] >= min_precision)
    if mask.any():
        return float(thresholds[mask][np.argmax(prec_arr[:-1][mask])])
    f1s = np.where((prec_arr + rec_arr) == 0, 0,
                   2*prec_arr*rec_arr / (prec_arr + rec_arr))
    return float(thresholds[np.argmax(f1s[:-1])])


def compute_full_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return dict(
        roc_auc   = round(roc_auc_score(y_true, y_prob),           4),
        pr_auc    = round(average_precision_score(y_true, y_prob),  4),
        recall    = round(recall_score(y_true, y_pred,    zero_division=0), 4),
        precision = round(precision_score(y_true, y_pred, zero_division=0), 4),
        f1        = round(f1_score(y_true, y_pred,        zero_division=0), 4),
        threshold = round(threshold, 4),
    )


def make_lgb(spw, n_toxic):
    return LGBMClassifier(
        n_estimators     = 400,
        max_depth        = 6,
        learning_rate    = 0.04,
        subsample        = 0.85,
        colsample_bytree = 0.75,
        scale_pos_weight = spw,
        min_child_samples= max(5, int(n_toxic * 0.01)),
        objective        = 'binary',
        metric           = 'average_precision',   # PR-AUC directly
        random_state     = 42,
        n_jobs           = -1,
        verbose          = -1,
    )


def make_cat(spw):
    return CatBoostClassifier(
        iterations        = 400,
        depth             = 6,
        learning_rate     = 0.04,
        auto_class_weights= 'Balanced',
        eval_metric       = 'PRAUC',              # PR-AUC directly
        random_seed       = 42,
        thread_count      = -1,
        verbose           = 0,
    )


def make_brf(n_toxic):
    return BalancedRandomForestClassifier(
        n_estimators      = 300,
        max_features      = 'sqrt',
        sampling_strategy = 'auto',
        replacement       = False,
        class_weight      = 'balanced_subsample',
        random_state      = 42,
        n_jobs            = -1,
    )


def make_rf(spw):
    return RandomForestClassifier(
        n_estimators = 300,
        max_features = 'sqrt',
        class_weight = 'balanced_subsample',      # resample per tree
        random_state = 42,
        n_jobs       = -1,
    )


def optimise_weights_pr_auc(y_true, p_xgb, p_m2, p_m3):
    """
    Find blend weights that maximise PR-AUC on OOF predictions.
    Weights are non-negative and sum to 1.
    """
    def neg_pr_auc(w):
        w = np.abs(w) / (np.abs(w).sum() + 1e-9)
        blend = w[0]*p_xgb + w[1]*p_m2 + w[2]*p_m3
        return -average_precision_score(y_true, blend)

    best_val, best_w = np.inf, np.array([0.5, 0.3, 0.2])
    # try multiple starting points to avoid local minima
    for x0 in [[0.6,0.2,0.2], [0.4,0.3,0.3],
                [0.2,0.5,0.3], [0.2,0.2,0.6],
                [1/3,1/3,1/3]]:
        res = minimize(neg_pr_auc, x0=x0, method='Nelder-Mead',
                       options={'maxiter': 300, 'xatol': 1e-5})
        if res.fun < best_val:
            best_val = res.fun
            best_w   = res.x

    w = np.abs(best_w) / np.abs(best_w).sum()
    return w


# ── Caches ────────────────────────────────────────────────────────
ensemble_results = []
lgb_models       = {}
rf_models        = {}
cat_models       = {}
brf_models       = {}
ensemble_models  = {}

# Store ensemble OOF for Stage 4 threshold tuning
ens_oof_probs  = {}   # ep → full OOF ensemble blend
ens_oof_labels = {}   # ep → y_ep

# ── Header ────────────────────────────────────────────────────────
hdr = (f"{'Endpoint':<22}{'Tier':<11}"
       f"{'XGB':>7}{'M2':>7}{'M3':>7}{'Ens':>7}  "
       f"{'PR-AUC':>7}{'Recall':>7}{'Precis':>7}{'F1':>7}  "
       f"{'W_xgb':>6}{'W_m2':>6}{'W_m3':>6}")
print("STAGE 3 — Correlation-Aware Ensemble  |  weights optimised for PR-AUC")
print(hdr)
print("─" * len(hdr))


# ════════════════════════════════════════════════════════
# MAIN LOOP
# ════════════════════════════════════════════════════════
for ep in toxicity_labels:
    if ep not in ep_data:
        continue

    # use augmented X (includes cross-endpoint OOF meta-features from Stage 2)
    mask, X_ep, y_ep, spw = ep_data_aug.get(ep, ep_data[ep])
    n_toxic  = int((y_ep == 1).sum())

    cached_xgb_preds = xgb_fold_preds[ep]
    cached_xgb_aucs  = xgb_fold_aucs[ep]
    final_xgb2       = xgb_models[ep]

    is_special   = ep in SPECIAL_ENDPOINTS
    is_rf_strong = ep in RF_STRONG_ENDPOINTS
    has_corr     = len(CORRELATION_GROUPS.get(ep, [])) > 0

    if is_special:
        tier = 'CAT+BRF'
    elif is_rf_strong:
        tier = 'RF-heavy'
    else:
        tier = 'XGB-heavy'

    # ── per-fold OOF collectors ───────────────────────────────────
    oof_xgb_full = np.zeros(len(y_ep))
    oof_m2_full  = np.zeros(len(y_ep))
    oof_m3_full  = np.zeros(len(y_ep))

    fold_roc_xgb = []
    fold_roc_m2  = []
    fold_roc_m3  = []

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_ep, y_ep)):
        X_tr, X_val = X_ep[tr_idx], X_ep[val_idx]
        y_tr, y_val = y_ep[tr_idx], y_ep[val_idx]

        # XGB — from cache, no retraining
        p_xgb = cached_xgb_preds[val_idx]
        oof_xgb_full[val_idx] = p_xgb

        # ── SMOTE on train fold for M2 and M3 ────────────────────
        n_tox_fold = int((y_tr == 1).sum())
        if HAS_SMOTE and n_tox_fold >= 6:
            try:
                from imblearn.over_sampling import BorderlineSMOTE
                k = min(5, n_tox_fold - 1)
                ratio = min(0.45, (n_tox_fold * 3) / (y_tr == 0).sum())
                X_tr_s, y_tr_s = BorderlineSMOTE(
                    sampling_strategy=ratio,
                    k_neighbors=k, random_state=42
                ).fit_resample(X_tr, y_tr)
            except Exception:
                X_tr_s, y_tr_s = X_tr, y_tr
        else:
            X_tr_s, y_tr_s = X_tr, y_tr

        # ── Model 2: CatBoost (special) or LightGBM ──────────────
        if is_special and HAS_CAT:
            _m2 = make_cat(spw)
            _m2.fit(X_tr, y_tr)          # CatBoost handles imbalance internally
        elif HAS_LGB:
            _m2 = make_lgb(spw, n_toxic)
            _m2.fit(X_tr_s, y_tr_s)
        else:
            _m2 = None

        p_m2 = _m2.predict_proba(X_val)[:, 1] if _m2 else p_xgb
        oof_m2_full[val_idx] = p_m2

        # ── Model 3: BRF (special) or balanced RF ────────────────
        if is_special and HAS_BRF:
            _m3 = make_brf(n_toxic)
            _m3.fit(X_tr, y_tr)          # BRF balances internally
        else:
            _m3 = make_rf(spw)
            _m3.fit(X_tr_s, y_tr_s)

        p_m3 = _m3.predict_proba(X_val)[:, 1]
        oof_m3_full[val_idx] = p_m3

        fold_roc_xgb.append(roc_auc_score(y_val, p_xgb))
        fold_roc_m2.append(roc_auc_score(y_val, p_m2))
        fold_roc_m3.append(roc_auc_score(y_val, p_m3))

    # ── Optimise blend weights on full OOF for PR-AUC ────────────
    opt_w = optimise_weights_pr_auc(
        y_ep, oof_xgb_full, oof_m2_full, oof_m3_full
    )

    ens_oof = opt_w[0]*oof_xgb_full + opt_w[1]*oof_m2_full + opt_w[2]*oof_m3_full

    # Write ensemble OOF back — Stage 4 will tune threshold on this
    ens_oof_probs[ep]  = ens_oof
    ens_oof_labels[ep] = y_ep
    # also update oof_probs so Stage 4 picks up ensemble (not XGB-only)
    oof_probs[ep]      = ens_oof
    oof_labels[ep]     = y_ep

    # ── Compute full metrics on OOF ensemble ─────────────────────
    best_thr = tune_threshold_recall_precision(
        y_ep, ens_oof, min_recall=0.65, min_precision=0.35
    )
    m = compute_full_metrics(y_ep, ens_oof, best_thr)

    auc_xgb = float(np.mean(fold_roc_xgb))
    auc_m2  = float(np.mean(fold_roc_m2))
    auc_m3  = float(np.mean(fold_roc_m3))

    # ── Train final models on full endpoint data ──────────────────
    if is_special and HAS_CAT:
        final_m2 = make_cat(spw)
        final_m2.fit(X_ep, y_ep)
        cat_models[ep] = final_m2
    elif HAS_LGB:
        final_m2 = make_lgb(spw, n_toxic)
        final_m2.fit(X_ep, y_ep)
        lgb_models[ep] = final_m2
    else:
        final_m2 = None

    if is_special and HAS_BRF:
        final_m3 = make_brf(n_toxic)
        final_m3.fit(X_ep, y_ep)
        brf_models[ep] = final_m3
    else:
        final_m3 = make_rf(spw)
        final_m3.fit(X_ep, y_ep)
        rf_models[ep] = final_m3

    ensemble_models[ep] = {
        'xgb'        : final_xgb2,
        'model2'     : final_m2,
        'model3'     : final_m3,
        'weights'    : opt_w.tolist(),   # optimised, not hardcoded
        'tier'       : tier,
        'threshold'  : best_thr,
        'is_special' : is_special,
        'has_corr'   : has_corr,
    }

    ensemble_results.append({
        'endpoint'    : ep,
        'tier'        : tier,
        'n_rows'      : len(y_ep),
        'n_toxic'     : n_toxic,
        'xgb_auc'     : round(auc_xgb,      4),
        'm2_auc'      : round(auc_m2,        4),
        'm3_auc'      : round(auc_m3,        4),
        'ensemble_auc': round(m['roc_auc'],  4),
        'pr_auc'      : round(m['pr_auc'],   4),
        'recall'      : round(m['recall'],   4),
        'precision'   : round(m['precision'],4),
        'f1'          : round(m['f1'],       4),
        'threshold'   : round(best_thr,      4),
        'delta_vs_xgb': round(m['roc_auc'] - auc_xgb, 4),
        'w_xgb'       : round(opt_w[0],     3),
        'w_m2'        : round(opt_w[1],     3),
        'w_m3'        : round(opt_w[2],     3),
    })

    print(f"{ep:<22}{tier:<11}"
          f"{auc_xgb:>7.4f}{auc_m2:>7.4f}{auc_m3:>7.4f}{m['roc_auc']:>7.4f}  "
          f"{m['pr_auc']:>7.4f}{m['recall']:>7.4f}{m['precision']:>7.4f}{m['f1']:>7.4f}  "
          f"{opt_w[0]:>6.2f}{opt_w[1]:>6.2f}{opt_w[2]:>6.2f}")


# ════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════
ens_df    = pd.DataFrame(ensemble_results)
mean_xgb2 = ens_df['xgb_auc'].mean()
mean_ens  = ens_df['ensemble_auc'].mean()

numeric_cols = ['ensemble_auc', 'pr_auc', 'recall', 'precision', 'f1']

print("\n" + "═" * len(hdr))
print("MACRO AVERAGE  (unweighted)")
print("═" * len(hdr))
for c in numeric_cols:
    print(f"  {c:<14}  mean={ens_df[c].mean():.4f}   std={ens_df[c].std():.4f}")

print("\nWEIGHTED AVERAGE  (by n_toxic)")
w = ens_df['n_toxic'].values
for c in numeric_cols:
    print(f"  {c:<14}  weighted_mean={np.average(ens_df[c], weights=w):.4f}")

print(f"\n  XGB alone mean ROC-AUC : {mean_xgb2:.4f}")
print(f"  Ensemble mean ROC-AUC  : {mean_ens:.4f}")
print(f"  Improvement            : {mean_ens - mean_xgb2:+.4f}")

# Per-tier breakdown
print()
for t in ['CAT+BRF', 'RF-heavy', 'XGB-heavy']:
    sub = ens_df[ens_df['tier'] == t]
    if len(sub) == 0:
        continue
    print(f"  {t:<12} → ROC={sub['ensemble_auc'].mean():.4f}  "
          f"PR={sub['pr_auc'].mean():.4f}  "
          f"Recall={sub['recall'].mean():.4f}  "
          f"Prec={sub['precision'].mean():.4f}  "
          f"F1={sub['f1'].mean():.4f}")

# Flag struggling endpoints
struggling = ens_df[(ens_df['recall'] < 0.60) | (ens_df['pr_auc'] < 0.50)]
if not struggling.empty:
    print(f"\n⚠  Still below target (recall<0.60 or PR-AUC<0.50):")
    print(struggling[['endpoint','tier','n_toxic','pr_auc',
                       'recall','precision','f1']].to_string(index=False))

print("\nFull results:")
print(ens_df[['endpoint','tier','n_toxic',
              'ensemble_auc','pr_auc','recall','precision','f1',
              'threshold','w_xgb','w_m2','w_m3']].to_string(index=False))

STAGE 3 — Correlation-Aware Ensemble  |  weights optimised for PR-AUC
Endpoint              Tier           XGB     M2     M3    Ens   PR-AUC Recall Precis     F1   W_xgb  W_m2  W_m3
───────────────────────────────────────────────────────────────────────────────────────────────────────────────
NR-AR                 CAT+BRF     0.8014 0.8098 0.7930 0.8080   0.5559 0.4903 0.8207 0.6138    0.51  0.49  0.00
NR-AR-LBD             CAT+BRF     0.8804 0.8892 0.8703 0.8919   0.6756 0.6540 0.5576 0.6019    0.42  0.54  0.03
NR-AhR                XGB-heavy   0.9089 0.8973 0.9109 0.9090   0.6825 0.6510 0.6046 0.6270    0.93  0.03  0.04
NR-Aromatase          XGB-heavy   0.8909 0.8788 0.8801 0.8946   0.5293 0.4600 0.5287 0.4920    0.41  0.43  0.16
NR-ER                 CAT+BRF     0.7567 0.7500 0.7458 0.7571   0.5147 0.4311 0.5491 0.4830    0.55  0.21  0.24
NR-ER-LBD             RF-heavy    0.8455 0.8437 0.8640 0.8647   0.5747 0.4871 0.6538 0.5583    0.18  0.29  0.53
NR-PPAR-gamma         RF-heavy    


## Stage 4 — Per-Endpoint Decision Threshold Optimisation


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STAGE 3 — Correlation-Aware Ensemble  |  PR-AUC + Recall focused
# ══════════════════════════════════════════════════════════════════
from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    recall_score, precision_score, f1_score,
    precision_recall_curve
)
from rdkit.Chem import MolFromSmarts

# ── Extra pharmacophore features for hard endpoints ───────────────
# Built inline — no featurisation cell change needed
NR_ER_EXTRA_SMARTS = {
    'smarts_bisphenol'    : MolFromSmarts('c1ccc(CCc2ccc(O)cc2)cc1'),
    'smarts_dihydroxy_aro': MolFromSmarts('c1cc(O)ccc1-c1ccc(O)cc1'),
    'smarts_stilbene'     : MolFromSmarts('c1ccc(/C=C/c2ccccc2)cc1'),
}

def compute_er_extra(mol):
    row = {}
    for name, patt in NR_ER_EXTRA_SMARTS.items():
        try:
            row[name] = len(mol.GetSubstructMatches(patt)) if patt else 0
        except Exception:
            row[name] = 0
    return row

er_extra_df = pd.DataFrame(
    [compute_er_extra(mol) for mol in mols],
    dtype=np.float32
)
print(f"NR-ER extra pharmacophore features : {er_extra_df.shape}")
print(er_extra_df.describe().round(3))

# Endpoints that get the extra features appended
ER_EXTRA_ENDPOINTS = {'NR-ER', 'NR-ER-LBD', 'NR-AR', 'NR-AR-LBD', 'NR-Aromatase'}

def get_X_for_ep(ep, mask, X_aug):
    """
    For ER-cluster endpoints, append the 3 extra pharmacophore features.
    For others, return X_aug as-is.
    """
    if ep not in ER_EXTRA_ENDPOINTS:
        return X_aug[mask]
    extra = er_extra_df.values[mask]
    return np.hstack([X_aug[mask], extra]).astype(np.float32)


# ── Correlation clusters ──────────────────────────────────────────
CORRELATION_GROUPS = {
    'NR-AR'        : ['NR-AR-LBD', 'NR-ER-LBD', 'NR-ER', 'NR-Aromatase'],
    'NR-AR-LBD'    : ['NR-AR',     'NR-ER-LBD', 'NR-ER'],
    'NR-ER'        : ['NR-ER-LBD', 'NR-Aromatase', 'NR-AR', 'NR-AR-LBD'],
    'NR-ER-LBD'    : ['NR-ER',     'NR-AR',     'NR-AR-LBD'],
    'NR-Aromatase' : ['NR-ER',     'NR-AR'],
    'SR-ATAD5'     : ['SR-p53',    'SR-MMP'],
    'SR-p53'       : ['SR-ATAD5',  'SR-MMP',    'SR-ARE'],
    'SR-MMP'       : ['SR-ARE',    'SR-ATAD5',  'SR-p53'],
    'SR-ARE'       : ['SR-MMP',    'SR-p53'],
    'NR-AhR'       : [],
    'NR-PPAR-gamma': [],
    'SR-HSE'       : [],
}

SPECIAL_ENDPOINTS   = {'NR-AR', 'NR-AR-LBD', 'NR-ER'}
RF_STRONG_ENDPOINTS = {'NR-PPAR-gamma', 'SR-ATAD5', 'SR-HSE', 'NR-ER-LBD'}

# Endpoints that get focal-style XGB due to poor PR-AUC
FOCAL_ENDPOINTS = {'NR-PPAR-gamma', 'SR-ATAD5', 'SR-HSE'}


# ── Model factories ───────────────────────────────────────────────
def make_lgb(spw, n_toxic):
    return LGBMClassifier(
        n_estimators     = 400,
        max_depth        = 6,
        learning_rate    = 0.04,
        subsample        = 0.85,
        colsample_bytree = 0.75,
        scale_pos_weight = spw,
        min_child_samples= max(5, int(n_toxic * 0.01)),
        objective        = 'binary',
        metric           = 'average_precision',
        random_state     = 42,
        n_jobs           = -1,
        verbose          = -1,
    )

def make_cat(spw):
    return CatBoostClassifier(
        iterations        = 400,
        depth             = 6,
        learning_rate     = 0.04,
        auto_class_weights= 'Balanced',
        eval_metric       = 'PRAUC',
        random_seed       = 42,
        thread_count      = -1,
        verbose           = 0,
    )

def make_brf(n_toxic):
    return BalancedRandomForestClassifier(
        n_estimators      = 300,
        max_features      = 'sqrt',
        sampling_strategy = 'auto',
        replacement       = False,
        class_weight      = 'balanced_subsample',
        random_state      = 42,
        n_jobs            = -1,
    )

def make_rf():
    return RandomForestClassifier(
        n_estimators = 300,
        max_features = 'sqrt',
        class_weight = 'balanced_subsample',
        random_state = 42,
        n_jobs       = -1,
    )

def make_xgb_focal(spw, n_toxic):
    """Focal-style XGB for hardest endpoints — slow LR, more trees, aggressive SPW."""
    return XGBClassifier(
        n_estimators      = 800,
        max_depth         = 4,           # shallower → less overfit on small minority
        learning_rate     = 0.01,        # very slow → careful minority learning
        subsample         = 0.9,
        colsample_bytree  = 0.7,
        min_child_weight  = 1,
        scale_pos_weight  = min(80, spw * 3),
        max_delta_step    = 1,
        grow_policy       = 'lossguide',
        max_leaves        = 31,
        eval_metric       = 'aucpr',
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = 0,
    )


# ── Helpers ───────────────────────────────────────────────────────
def smart_oversample_stage3(X_tr, y_tr):
    n_tox = int((y_tr == 1).sum())
    if not HAS_SMOTE or n_tox < 6:
        return X_tr, y_tr
    try:
        from imblearn.over_sampling import BorderlineSMOTE
        k     = min(5, n_tox - 1)
        ratio = min(0.45, (n_tox * 3) / (y_tr == 0).sum())
        return BorderlineSMOTE(
            sampling_strategy=ratio, k_neighbors=k, random_state=42
        ).fit_resample(X_tr, y_tr)
    except Exception:
        return X_tr, y_tr


def tune_threshold_recall_precision(y_true, y_prob,
                                     min_recall=0.65,
                                     min_precision=0.35):
    prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_prob)
    mask = (rec_arr[:-1] >= min_recall) & (prec_arr[:-1] >= min_precision)
    if mask.any():
        f1s = np.where(
            (prec_arr[:-1] + rec_arr[:-1]) == 0, 0,
            2*prec_arr[:-1]*rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1])
        )
        return float(thresholds[mask][np.argmax(f1s[mask])])
    # fallback: recall only
    mask_r = rec_arr[:-1] >= min_recall
    if mask_r.any():
        return float(thresholds[mask_r][np.argmax(prec_arr[:-1][mask_r])])
    # fallback: best F1
    f1s = np.where((prec_arr+rec_arr)==0, 0,
                   2*prec_arr*rec_arr/(prec_arr+rec_arr))
    return float(thresholds[np.argmax(f1s[:-1])])


def compute_full_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return dict(
        roc_auc   = round(roc_auc_score(y_true, y_prob),           4),
        pr_auc    = round(average_precision_score(y_true, y_prob),  4),
        recall    = round(recall_score(y_true, y_pred,    zero_division=0), 4),
        precision = round(precision_score(y_true, y_pred, zero_division=0), 4),
        f1        = round(f1_score(y_true, y_pred,        zero_division=0), 4),
        threshold = round(threshold, 4),
    )


def optimise_weights_pr_auc(y_true, p_xgb, p_m2, p_m3):
    def neg_pr_auc(w):
        w = np.abs(w) / (np.abs(w).sum() + 1e-9)
        return -average_precision_score(
            y_true, w[0]*p_xgb + w[1]*p_m2 + w[2]*p_m3
        )
    best_val, best_w = np.inf, np.array([1/3, 1/3, 1/3])
    for x0 in [[0.6,0.2,0.2],[0.4,0.3,0.3],
                [0.2,0.5,0.3],[0.2,0.2,0.6],[1/3,1/3,1/3]]:
        res = minimize(neg_pr_auc, x0=x0, method='Nelder-Mead',
                       options={'maxiter':300, 'xatol':1e-5})
        if res.fun < best_val:
            best_val, best_w = res.fun, res.x
    return np.abs(best_w) / np.abs(best_w).sum()


# ── Endpoint-specific threshold policies ─────────────────────────
THRESHOLD_POLICY = {
    'NR-AR'        : (0.65, 0.25),
    'NR-ER'        : (0.65, 0.25),
    'NR-ER-LBD'    : (0.65, 0.30),
    'NR-Aromatase' : (0.60, 0.25),
    'NR-PPAR-gamma': (0.55, 0.20),
    'SR-ATAD5'     : (0.60, 0.45),
    'SR-HSE'       : (0.55, 0.38),
    'NR-AR-LBD'    : (0.65, 0.35),
    'NR-AhR'       : (0.65, 0.35),
    'SR-ARE'       : (0.65, 0.35),
    'SR-MMP'       : (0.65, 0.50),
    'SR-p53'       : (0.65, 0.35),
}


# ── Caches ────────────────────────────────────────────────────────
ensemble_results = []
lgb_models       = {}
rf_models        = {}
cat_models       = {}
brf_models       = {}
ensemble_models  = {}
ens_oof_probs    = {}
ens_oof_labels   = {}

hdr = (f"{'Endpoint':<22}{'Tier':<11}"
       f"{'XGB':>7}{'M2':>7}{'M3':>7}{'Ens':>7}  "
       f"{'PR-AUC':>7}{'Recall':>7}{'Precis':>7}{'F1':>7}  "
       f"{'W_xgb':>6}{'W_m2':>6}{'W_m3':>6}")
print("STAGE 3 — Correlation-Aware Ensemble  |  weights optimised for PR-AUC")
print(hdr)
print("─" * len(hdr))


# ════════════════════════════════════════════════════════
# MAIN LOOP
# ════════════════════════════════════════════════════════
for ep in toxicity_labels:
    if ep not in ep_data:
        continue

    mask, _, y_ep, spw = ep_data_aug.get(ep, ep_data[ep])
    n_toxic = int((y_ep == 1).sum())

    # get X with extra ER pharmacophore features if applicable
    X_ep = get_X_for_ep(ep, mask, X_array_aug)

    cached_xgb_preds = xgb_fold_preds[ep]
    cached_xgb_aucs  = xgb_fold_aucs[ep]
    final_xgb2       = xgb_models[ep]

    is_special   = ep in SPECIAL_ENDPOINTS
    is_rf_strong = ep in RF_STRONG_ENDPOINTS
    is_focal     = ep in FOCAL_ENDPOINTS
    has_corr     = len(CORRELATION_GROUPS.get(ep, [])) > 0

    tier = ('CAT+BRF' if is_special else
            'RF-heavy' if is_rf_strong else 'XGB-heavy')

    oof_xgb_full = np.zeros(len(y_ep))
    oof_m2_full  = np.zeros(len(y_ep))
    oof_m3_full  = np.zeros(len(y_ep))
    fold_roc_xgb = []
    fold_roc_m2  = []
    fold_roc_m3  = []

    for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_ep, y_ep)):
        X_tr, X_val = X_ep[tr_idx], X_ep[val_idx]
        y_tr, y_val = y_ep[tr_idx], y_ep[val_idx]

        # XGB — from cache
        p_xgb = cached_xgb_preds[val_idx]
        oof_xgb_full[val_idx] = p_xgb

        # SMOTE on train fold for M2/M3
        X_tr_s, y_tr_s = smart_oversample_stage3(X_tr, y_tr)

        # ── Model 2 ──────────────────────────────────────────────
        if is_special and HAS_CAT:
            _m2 = make_cat(spw)
            _m2.fit(X_tr, y_tr)
        elif is_focal and HAS_LGB:
            # focal endpoints get aggressive LGB
            _m2 = LGBMClassifier(
                n_estimators     = 600,
                max_depth        = 4,
                learning_rate    = 0.02,
                subsample        = 0.9,
                colsample_bytree = 0.7,
                scale_pos_weight = min(80, spw * 3),
                min_child_samples= 1,
                objective        = 'binary',
                metric           = 'average_precision',
                random_state     = 42, n_jobs=-1, verbose=-1,
            )
            _m2.fit(X_tr_s, y_tr_s)
        elif HAS_LGB:
            _m2 = make_lgb(spw, n_toxic)
            _m2.fit(X_tr_s, y_tr_s)
        else:
            _m2 = None

        p_m2 = _m2.predict_proba(X_val)[:, 1] if _m2 else p_xgb
        oof_m2_full[val_idx] = p_m2

        # ── Model 3 ──────────────────────────────────────────────
        if is_focal:
            # focal endpoints get focal XGB as M3
            _m3 = make_xgb_focal(spw, n_toxic)
            _m3.fit(X_tr_s, y_tr_s)
        elif is_special and HAS_BRF:
            _m3 = make_brf(n_toxic)
            _m3.fit(X_tr, y_tr)
        else:
            _m3 = make_rf()
            _m3.fit(X_tr_s, y_tr_s)

        p_m3 = _m3.predict_proba(X_val)[:, 1]
        oof_m3_full[val_idx] = p_m3

        fold_roc_xgb.append(roc_auc_score(y_val, p_xgb))
        fold_roc_m2.append(roc_auc_score(y_val, p_m2))
        fold_roc_m3.append(roc_auc_score(y_val, p_m3))

    # ── Optimise blend weights on OOF ────────────────────────────
    opt_w = optimise_weights_pr_auc(
        y_ep, oof_xgb_full, oof_m2_full, oof_m3_full
    )
    ens_oof = (opt_w[0]*oof_xgb_full +
               opt_w[1]*oof_m2_full  +
               opt_w[2]*oof_m3_full)

    # write ensemble OOF back for Stage 4
    oof_probs[ep]      = ens_oof
    oof_labels[ep]     = y_ep
    ens_oof_probs[ep]  = ens_oof
    ens_oof_labels[ep] = y_ep

    # ── Threshold with endpoint-specific policy ───────────────────
    min_rec, min_prec = THRESHOLD_POLICY.get(ep, (0.65, 0.35))
    best_thr = tune_threshold_recall_precision(
        y_ep, ens_oof, min_recall=min_rec, min_precision=min_prec
    )
    m = compute_full_metrics(y_ep, ens_oof, best_thr)

    auc_xgb = float(np.mean(fold_roc_xgb))
    auc_m2  = float(np.mean(fold_roc_m2))
    auc_m3  = float(np.mean(fold_roc_m3))

    # ── Train final models on full data ───────────────────────────
    if is_special and HAS_CAT:
        final_m2 = make_cat(spw)
        final_m2.fit(X_ep, y_ep)
        cat_models[ep] = final_m2
    elif is_focal and HAS_LGB:
        final_m2 = LGBMClassifier(
            n_estimators=600, max_depth=4, learning_rate=0.02,
            subsample=0.9, colsample_bytree=0.7,
            scale_pos_weight=min(80, spw*3), min_child_samples=1,
            objective='binary', metric='average_precision',
            random_state=42, n_jobs=-1, verbose=-1,
        )
        final_m2.fit(X_ep, y_ep)
        lgb_models[ep] = final_m2
    elif HAS_LGB:
        final_m2 = make_lgb(spw, n_toxic)
        final_m2.fit(X_ep, y_ep)
        lgb_models[ep] = final_m2
    else:
        final_m2 = None

    if is_focal:
        final_m3 = make_xgb_focal(spw, n_toxic)
        final_m3.fit(X_ep, y_ep)
        rf_models[ep] = final_m3
    elif is_special and HAS_BRF:
        final_m3 = make_brf(n_toxic)
        final_m3.fit(X_ep, y_ep)
        brf_models[ep] = final_m3
    else:
        final_m3 = make_rf()
        final_m3.fit(X_ep, y_ep)
        rf_models[ep] = final_m3

    ensemble_models[ep] = {
        'xgb'      : final_xgb2,
        'model2'   : final_m2,
        'model3'   : final_m3,
        'weights'  : opt_w.tolist(),
        'tier'     : tier,
        'threshold': best_thr,
        'is_special': is_special,
        'is_focal'  : is_focal,
        'er_extra'  : ep in ER_EXTRA_ENDPOINTS,
    }

    ensemble_results.append({
        'endpoint'    : ep, 'tier': tier,
        'n_rows'      : len(y_ep), 'n_toxic': n_toxic,
        'xgb_auc'     : round(auc_xgb,       4),
        'm2_auc'      : round(auc_m2,         4),
        'm3_auc'      : round(auc_m3,         4),
        'ensemble_auc': round(m['roc_auc'],   4),
        'pr_auc'      : round(m['pr_auc'],    4),
        'recall'      : round(m['recall'],    4),
        'precision'   : round(m['precision'], 4),
        'f1'          : round(m['f1'],        4),
        'threshold'   : round(best_thr,       4),
        'delta_vs_xgb': round(m['roc_auc'] - auc_xgb, 4),
        'w_xgb'       : round(opt_w[0], 3),
        'w_m2'        : round(opt_w[1], 3),
        'w_m3'        : round(opt_w[2], 3),
    })

    print(f"{ep:<22}{tier:<11}"
          f"{auc_xgb:>7.4f}{auc_m2:>7.4f}{auc_m3:>7.4f}{m['roc_auc']:>7.4f}  "
          f"{m['pr_auc']:>7.4f}{m['recall']:>7.4f}{m['precision']:>7.4f}{m['f1']:>7.4f}  "
          f"{opt_w[0]:>6.2f}{opt_w[1]:>6.2f}{opt_w[2]:>6.2f}")


# ════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════
ens_df    = pd.DataFrame(ensemble_results)
mean_xgb2 = ens_df['xgb_auc'].mean()
mean_ens  = ens_df['ensemble_auc'].mean()
numeric_cols = ['ensemble_auc', 'pr_auc', 'recall', 'precision', 'f1']

print("\n" + "═"*len(hdr))
print("MACRO AVERAGE  (unweighted)")
print("═"*len(hdr))
for c in numeric_cols:
    print(f"  {c:<14}  mean={ens_df[c].mean():.4f}   std={ens_df[c].std():.4f}")

print("\nWEIGHTED AVERAGE  (by n_toxic)")
w = ens_df['n_toxic'].values
for c in numeric_cols:
    print(f"  {c:<14}  weighted_mean={np.average(ens_df[c], weights=w):.4f}")

print(f"\n  XGB alone  : {mean_xgb2:.4f}")
print(f"  Ensemble   : {mean_ens:.4f}")
print(f"  Improvement: {mean_ens-mean_xgb2:+.4f}")

for t in ['CAT+BRF', 'RF-heavy', 'XGB-heavy']:
    sub = ens_df[ens_df['tier']==t]
    if len(sub) == 0: continue
    print(f"  {t:<12} → ROC={sub['ensemble_auc'].mean():.4f}  "
          f"PR={sub['pr_auc'].mean():.4f}  "
          f"Recall={sub['recall'].mean():.4f}  "
          f"Prec={sub['precision'].mean():.4f}  "
          f"F1={sub['f1'].mean():.4f}")

struggling = ens_df[(ens_df['recall'] < 0.55) | (ens_df['pr_auc'] < 0.45)]
if not struggling.empty:
    print(f"\n⚠  Still below target:")
    print(struggling[['endpoint','tier','n_toxic','pr_auc',
                       'recall','precision','f1']].to_string(index=False))
else:
    print("\n✓  All endpoints meet targets")

print("\nFull results:")
print(ens_df[['endpoint','tier','n_toxic','ensemble_auc',
              'pr_auc','recall','precision','f1',
              'threshold','w_xgb','w_m2','w_m3']].to_string(index=False))

NR-ER extra pharmacophore features : (7823, 3)
       smarts_bisphenol  smarts_dihydroxy_aro  smarts_stilbene
count          7823.000              7823.000         7823.000
mean              0.006                 0.001            0.006
std               0.122                 0.023            0.090
min               0.000                 0.000            0.000
25%               0.000                 0.000            0.000
50%               0.000                 0.000            0.000
75%               0.000                 0.000            0.000
max               4.000                 1.000            2.000
STAGE 3 — Correlation-Aware Ensemble  |  weights optimised for PR-AUC
Endpoint              Tier           XGB     M2     M3    Ens   PR-AUC Recall Precis     F1   W_xgb  W_m2  W_m3
───────────────────────────────────────────────────────────────────────────────────────────────────────────────
NR-AR                 CAT+BRF     0.8014 0.8123 0.7905 0.8122   0.5615 0.6526 0.1534 0.2485 

In [ ]:

from sklearn.metrics import roc_curve, recall_score, precision_score, f1_score

optimal_thresholds = {}
threshold_results  = []

print(f'{"Endpoint":<22} {"Youden thr":>10} {"Recall@0.5":>10} '
      f'{"Recall@J":>9} {"Gain":>7}')
print('-' * 65)

for ep in oof_probs:
    y_true = oof_labels[ep]
    y_prob = oof_probs[ep]

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_scores  = tpr - fpr
    best_idx  = int(np.argmax(j_scores))
    best_thr  = float(thresholds[best_idx])
    optimal_thresholds[ep] = best_thr

    recall_05  = recall_score(y_true, (y_prob >= 0.5).astype(int),       zero_division=0)
    recall_opt = recall_score(y_true, (y_prob >= best_thr).astype(int),  zero_division=0)
    gain = recall_opt - recall_05

    threshold_results.append({
        'endpoint'   : ep,
        'youden_thr' : round(best_thr, 3),
        'recall_050' : round(recall_05,  3),
        'recall_opt' : round(recall_opt, 3),
        'recall_gain': round(gain, 3),
    })
    marker = '  ← significant gain' if gain > 0.10 else ''
    print(f'{ep:<22} {best_thr:>10.3f} {recall_05:>10.3f} '
          f'{recall_opt:>9.3f} {gain:>+7.3f}{marker}')

thr_df = pd.DataFrame(threshold_results)
thr_df.to_csv('results/optimal_thresholds.csv', index=False)
print(f'\nMean recall gain from threshold optimisation : '
      f'{thr_df["recall_gain"].mean():+.3f}')

with open('models/optimal_thresholds.json', 'w') as f:
    json.dump(optimal_thresholds, f, indent=2)
print('Saved → results/optimal_thresholds.csv')
print('Saved → models/optimal_thresholds.json')


## Result Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax  = axes[0]
x   = np.arange(len(ens_df))
w   = 0.35

bars_ens = ax.bar(x - w/2, ens_df['ensemble_auc'], w,
                  color='#639922', alpha=0.85, label='Ensemble')
bars_xgb = ax.bar(x + w/2, ens_df['xgb_auc'],     w,
                  color='#EF9F27', alpha=0.85, label='XGBoost alone')

for bar, val in zip(bars_ens, ens_df['ensemble_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)
for bar, val in zip(bars_xgb, ens_df['xgb_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)

ax.axhline(mean_ens,  color='#3B6D11', linestyle='--', lw=1.5,
           label=f'Ensemble mean {mean_ens:.4f}')
ax.axhline(mean_xgb2, color='#854F0B', linestyle=':',  lw=1.5,
           label=f'XGB mean {mean_xgb2:.4f}')
ax.set_xticks(x)
ax.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.65, 1.02)
ax.set_title('Ensemble vs XGBoost alone', fontsize=11)
ax.legend(fontsize=8)
ax.yaxis.grid(True, alpha=0.3)

ax2 = axes[1]
colors = ['#3B6D11' if d > 0 else '#A32D2D' for d in ens_df['delta_vs_xgb']]
bars_d = ax2.bar(x, ens_df['delta_vs_xgb'], color=colors, alpha=0.85)
for bar, val in zip(bars_d, ens_df['delta_vs_xgb']):
    ypos = bar.get_height()+0.001 if val >= 0 else bar.get_height()-0.004
    ax2.text(bar.get_x()+bar.get_width()/2, ypos,
             f'{val:+.4f}', ha='center', fontsize=8)
ax2.axhline(0, color='black', lw=1, alpha=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('AUC gain (Ensemble − XGB alone)')
ax2.set_title('AUC improvement per endpoint', fontsize=11)
ax2.yaxis.grid(True, alpha=0.3)

plt.suptitle(f'Ensemble Results: {mean_xgb2:.4f} → {mean_ens:.4f} '
             f'({mean_ens-mean_xgb2:+.4f})', fontsize=12)
plt.tight_layout()
plt.savefig('plots/ensemble_results.png', dpi=150)
plt.show()
print('Saved → plots/ensemble_results.png')

# ROC curves
fig2, ax3 = plt.subplots(figsize=(9, 7))
for i, ep in enumerate(oof_probs):
    fpr, tpr, _ = roc_curve(oof_labels[ep], oof_probs[ep])
    row     = ens_df[ens_df['endpoint']==ep]
    ens_auc = float(row['ensemble_auc'].values[0]) if len(row)>0 else 0
    ax3.plot(fpr, tpr, color=COLORS[i % len(COLORS)], lw=1.5,
             label=f'{ep}  ({ens_auc:.3f})')
ax3.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title(f'ROC Curves — Ensemble\nMean AUC = {mean_ens:.4f}')
ax3.legend(loc='lower right', fontsize=8, ncol=2)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves_ensemble.png', dpi=150)
plt.show()
print('Saved → plots/roc_curves_ensemble.png')


---
## Stage 5 — Feature Importance Analysis


In [ ]:

importance_records = []

print('Computing feature importances per endpoint...')
for ep in toxicity_labels:
    if ep not in xgb_models:
        continue
    model    = xgb_models[ep]
    imp_dict = model.get_booster().get_score(importance_type='gain')

    
    for fname, score in imp_dict.items():
        try:
            idx = int(fname[1:])   # strip 'f' prefix
            col = feat_cols[idx] if idx < len(feat_cols) else fname
        except (ValueError, IndexError):
            col = fname
        importance_records.append({
            'endpoint': ep,
            'feature' : col,
            'gain'    : score,
        })

imp_df = pd.DataFrame(importance_records)

# ── Global top-20 features across all endpoints ──────────────────
global_imp = (imp_df.groupby('feature')['gain']
              .mean()
              .sort_values(ascending=False)
              .head(20))

print(f'\nTop 20 features by mean gain across all endpoints:')
print(f'{"Feature":<35} {"Mean Gain":>10}')
print('-' * 48)
for feat, gain in global_imp.items():
    # Classify feature type
    if feat.startswith('morgan'):
        ftype = '[ECFP fingerprint]'
    elif feat.startswith('ap_'):
        ftype = '[Atom Pair fp]'
    elif feat.startswith('tt_'):
        ftype = '[Topo Torsion fp]'
    elif feat.startswith('maccs'):
        ftype = '[MACCS key]'
    elif feat.startswith('smarts'):
        ftype = '[SMARTS fragment]'
    elif feat.startswith('oof_'):
        ftype = '[cross-endpoint]'
    elif feat.startswith('autocorr') or feat.startswith('whim'):
        ftype = '[3D descriptor]'
    elif feat in ['MW','logP','QED','TPSA','HBD','HBA','RotBonds',
                  'ArRings','FracCSP3','HeavyAtoms']:
        ftype = '[physicochemical]'
    else:
        ftype = '[descriptor]'
    print(f'{feat:<35} {gain:>10.2f}  {ftype}')

# ── Per-endpoint top-5 features ──────────────────────────────────
print(f'\n{"="*65}')
print('Top 5 features per endpoint:')
print(f'{"="*65}')
for ep in toxicity_labels:
    ep_imp = (imp_df[imp_df['endpoint']==ep]
              .sort_values('gain', ascending=False)
              .head(5))
    if len(ep_imp) == 0:
        continue
    print(f'\n{ep}:')
    for _, row in ep_imp.iterrows():
        print(f'  {row["feature"]:<35} gain={row["gain"]:.2f}')

# ── Feature category breakdown ────────────────────────────────────
def categorise(feat):
    if feat.startswith('morgan'):   return 'Morgan fingerprint'
    if feat.startswith('ap_'):      return 'Atom Pair fp'
    if feat.startswith('tt_'):      return 'Topo Torsion fp'
    if feat.startswith('maccs'):    return 'MACCS keys'
    if feat.startswith('smarts'):   return 'SMARTS fragments'
    if feat.startswith('oof_'):     return 'Cross-endpoint OOF'
    if feat.startswith('autocorr'): return '3D AUTOCORR'
    if feat.startswith('whim'):     return '3D WHIM'
    if feat in ['MW','logP','QED','TPSA','HBD','HBA',
                'RotBonds','ArRings','FracCSP3','HeavyAtoms']:
        return 'Core physicochemical'
    if feat.startswith('lip_') or feat in ['logP_x_MW','TPSA_per_MW',
                                            'HBD_HBA_sum','logMW']:
        return 'Engineered features'
    if feat.startswith('chiral') or feat.startswith('total_charge'):
        return 'Chirality/charge'
    return 'Extended descriptors'

imp_df['category'] = imp_df['feature'].apply(categorise)
cat_summary = (imp_df.groupby('category')['gain']
               .agg(['mean','count'])
               .sort_values('mean', ascending=False))
cat_summary.columns = ['Mean Gain', 'N Features Used']

print(f'\n{"="*55}')
print('Feature category contribution summary:')
print(f'{"="*55}')
print(cat_summary.round(2).to_string())

imp_df.to_csv('results/feature_importances.csv', index=False)
cat_summary.to_csv('results/feature_category_summary.csv')
print('\nSaved → results/feature_importances.csv')
print('Saved → results/feature_category_summary.csv')

# ── Importance bar chart ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Global top-20
ax = axes[0]
feat_names = [f[:28]+'…' if len(f)>28 else f for f in global_imp.index]
bars = ax.barh(range(len(global_imp)), global_imp.values,
               color=COLORS[:len(global_imp)] if len(global_imp)<=len(COLORS)
               else ['#378ADD']*len(global_imp), alpha=0.85)
ax.set_yticks(range(len(global_imp)))
ax.set_yticklabels(feat_names, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Mean XGBoost Gain')
ax.set_title('Top 20 Features Linked to Toxicity\n(mean gain across all endpoints)', fontsize=10)
ax.xaxis.grid(True, alpha=0.3)

# Category pie
ax2 = axes[1]
cat_vals = cat_summary['Mean Gain'].values
cat_labs = [f'{idx}\n({v:.0f})' for idx, v in zip(cat_summary.index, cat_vals)]
ax2.pie(cat_vals, labels=cat_labs, colors=COLORS[:len(cat_vals)],
        autopct='%1.1f%%', startangle=140, textprops={'fontsize': 8})
ax2.set_title('Feature Category Contribution\n(by mean XGBoost gain)', fontsize=10)

plt.suptitle('Structural Features Linked to Toxicity — Feature Importance Analysis',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/feature_importance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/feature_importance_analysis.png')


---
## Stage 5 — Prediction Interface for New Compounds

A standalone function that accepts a SMILES string and returns toxicity probability
and binary prediction for all 12 Tox21 endpoints, using the optimal per-endpoint
decision threshold computed in Stage 3.

This addresses the hackathon requirement: *"build a simple prediction interface
or visualisation for evaluating new compounds"*.

In [ ]:
# ════════════════════════════════════════════════════════════════
# STAGE 5 — Prediction Interface
# Accepts any SMILES string → returns per-endpoint toxicity probs
# and binary predictions using the optimal Youden J thresholds.
# ════════════════════════════════════════════════════════════════

def build_features_for_smiles(smiles_str):
    """Compute the full feature vector for a single SMILES string.
    Returns a (1, n_features) numpy array matching feat_cols order,
    or None if the SMILES is invalid."""
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return None

    # ── Fingerprints ─────────────────────────────────────────────
    morgan_gen_ = rdFingerprintGenerator.GetMorganGenerator(
        radius=2, fpSize=2048, includeChirality=True)
    morgan_r2 = list(morgan_gen_.GetFingerprint(mol))

    ap_fp = list(rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True))
    tt_fp = list(rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(
        mol, nBits=1024, includeChirality=True))
    maccs_fp = list(MACCSkeys.GenMACCSKeys(mol))

    fp_raw = morgan_r2 + ap_fp + tt_fp + maccs_fp
    # Apply same VarianceThreshold fitted on training data
    fp_kept = np.array(fp_raw)[vt.get_support()].tolist()

    # ── Continuous descriptors ────────────────────────────────────
    desc_row  = compute_descriptors(mol)
    ext_row   = compute_extended_descriptors(mol)
    smarts_row= compute_smarts_features(mol)
    chiral_row= compute_chirality_charge(mol)
    zinc_row  = compute_zinc_features(mol)

    # Engineered features
    mw_   = desc_row.get('MW', np.nan)
    logp_ = desc_row.get('logP', np.nan)
    tpsa_ = desc_row.get('TPSA', np.nan)
    hbd_  = desc_row.get('HBD', np.nan)
    hba_  = desc_row.get('HBA', np.nan)
    eng_row = {
        'lip_pass'      : int(mw_<500 and logp_<5 and hbd_<=5 and hba_<=10),
        'lip_violations': int(mw_>=500)+int(logp_>=5)+int(hbd_>5)+int(hba_>10),
        'logP_x_MW'     : logp_ * mw_,
        'TPSA_per_MW'   : tpsa_ / (mw_ + 1),
        'HBD_HBA_sum'   : hbd_ + hba_,
        'logMW'         : np.log1p(mw_),
    }

    # Combine all continuous columns in training order
    cont_row = {}
    for col in continuous_df.columns:
        if col in desc_row:    cont_row[col] = desc_row[col]
        elif col in ext_row:   cont_row[col] = ext_row[col]
        elif col in smarts_row:cont_row[col] = smarts_row[col]
        elif col in eng_row:   cont_row[col] = eng_row[col]
        elif col in zinc_row:  cont_row[col] = zinc_row[col]
        elif col in chiral_row:cont_row[col] = chiral_row[col]
        else:                  cont_row[col] = np.nan

    cont_vals = np.array([cont_row.get(c, np.nan) for c in continuous_df.columns],
                         dtype=np.float32)
    cont_vals = np.nan_to_num(cont_vals, nan=0.0, posinf=0.0, neginf=0.0)

    row = np.concatenate([np.array(fp_kept, dtype=np.float32), cont_vals])
    return row.reshape(1, -1)


def predict_toxicity(smiles_str, verbose=True):
    """
    Predict Tox21 toxicity for a SMILES string.

    Returns a DataFrame with columns:
        endpoint, probability, prediction, threshold, risk_level
    """
    X_new = build_features_for_smiles(smiles_str)
    if X_new is None:
        print(f'Invalid SMILES: {smiles_str}')
        return None

    results = []
    for ep in toxicity_labels:
        if ep not in xgb_models:
            continue
        model = ensemble_models.get(ep)
        if model is None:
            prob = xgb_models[ep].predict_proba(X_new)[0, 1]
        else:
            w   = model['weights']
            p_x = model['xgb'].predict_proba(X_new)[0, 1]
            p_m2 = model['model2'].predict_proba(X_new)[0, 1] if model['model2'] else p_x
            p_m3 = model['model3'].predict_proba(X_new)[0, 1] if model['model3'] else p_x
            prob = w[0]*p_x + w[1]*p_m2 + w[2]*p_m3

        thr  = optimal_thresholds.get(ep, 0.5)
        pred = int(prob >= thr)
        risk = 'HIGH' if prob > 0.7 else ('MODERATE' if prob > 0.4 else 'LOW')

        results.append({
            'endpoint'   : ep,
            'probability': round(float(prob), 4),
            'prediction' : 'TOXIC' if pred == 1 else 'SAFE',
            'threshold'  : round(thr, 3),
            'risk_level' : risk,
        })

    result_df = pd.DataFrame(results)

    if verbose:
        print(f'\nToxicity Prediction for: {smiles_str}')
        print('=' * 65)
        print(f'{"Endpoint":<22} {"Prob":>6} {"Pred":>7} {"Risk":>10}  {"Thr":>6}')
        print('-' * 55)
        for _, row in result_df.iterrows():
            marker = ' ◄' if row['prediction'] == 'TOXIC' else ''
            print(f'{row["endpoint"]:<22} {row["probability"]:>6.4f} '
                  f'{row["prediction"]:>7} {row["risk_level"]:>10}  '
                  f'{row["threshold"]:>6.3f}{marker}')
        n_toxic = (result_df['prediction'] == 'TOXIC').sum()
        print(f'\nSummary: {n_toxic}/{len(result_df)} endpoints predicted TOXIC')
        mean_prob = result_df['probability'].mean()
        print(f'Mean toxicity probability across all endpoints: {mean_prob:.4f}')
    return result_df


# ── Visualisation helper ──────────────────────────────────────────
def visualise_prediction(smiles_str):
    """Bar chart of predicted toxicity probabilities across all endpoints."""
    result_df = predict_toxicity(smiles_str, verbose=False)
    if result_df is None:
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    colors_ = ['#A32D2D' if p == 'TOXIC' else '#3B6D11'
               for p in result_df['prediction']]
    bars = ax.bar(result_df['endpoint'], result_df['probability'],
                  color=colors_, alpha=0.85, edgecolor='white', linewidth=0.5)
    # Threshold markers
    for i, (_, row) in enumerate(result_df.iterrows()):
        ax.plot([i-0.4, i+0.4], [row['threshold'], row['threshold']],
                'k--', lw=1.2, alpha=0.6)
    ax.set_xticks(range(len(result_df)))
    ax.set_xticklabels(result_df['endpoint'], rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Predicted Toxicity Probability')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'Toxicity Profile: {smiles_str[:60]}', fontsize=10)
    ax.yaxis.grid(True, alpha=0.3)

    from matplotlib.patches import Patch
    legend_elems = [Patch(facecolor='#A32D2D', label='Predicted TOXIC'),
                    Patch(facecolor='#3B6D11', label='Predicted SAFE'),
                    plt.Line2D([0],[0], color='black', linestyle='--',
                               label='Endpoint threshold')]
    ax.legend(handles=legend_elems, fontsize=9)
    plt.tight_layout()
    plt.savefig('plots/prediction_profile.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → plots/prediction_profile.png')


# ── Demo: run on 3 example compounds ─────────────────────────────
print('PREDICTION INTERFACE — DEMO')
print('=' * 65)

DEMO_COMPOUNDS = {
    'Bisphenol A (known endocrine disruptor)' :
        'CC(C)(c1ccc(O)cc1)c1ccc(O)cc1',
    'Caffeine (generally safe)' :
        'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
    'Tamoxifen (ER antagonist)' :
        'CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1',
}

for name, smi in DEMO_COMPOUNDS.items():
    print(f'\n--- {name} ---')
    res = predict_toxicity(smi, verbose=True)

print('\n--- Visualising Bisphenol A ---')
visualise_prediction('CC(C)(c1ccc(O)cc1)c1ccc(O)cc1')
print('\nTo evaluate your own compound:')
print('  result = predict_toxicity("YOUR_SMILES_HERE")')
print('  visualise_prediction("YOUR_SMILES_HERE")')


---
## Results — ROC-AUC Comparison Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax  = axes[0]
x   = np.arange(len(ens_df))
w   = 0.35

bars_ens = ax.bar(x - w/2, ens_df['ensemble_auc'], w,
                  color='#639922', alpha=0.85, label='Ensemble')
bars_xgb = ax.bar(x + w/2, ens_df['xgb_auc'],     w,
                  color='#EF9F27', alpha=0.85, label='XGBoost alone')

for bar, val in zip(bars_ens, ens_df['ensemble_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)
for bar, val in zip(bars_xgb, ens_df['xgb_auc']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', fontsize=7, rotation=90)

ax.axhline(mean_ens,  color='#3B6D11', linestyle='--', lw=1.5,
           label=f'Ensemble mean {mean_ens:.4f}')
ax.axhline(mean_xgb2, color='#854F0B', linestyle=':',  lw=1.5,
           label=f'XGB mean {mean_xgb2:.4f}')
ax.set_xticks(x)
ax.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.65, 1.02)
ax.set_title('Ensemble vs XGBoost alone', fontsize=11)
ax.legend(fontsize=8)
ax.yaxis.grid(True, alpha=0.3)

ax2 = axes[1]
colors = ['#3B6D11' if d > 0 else '#A32D2D' for d in ens_df['delta_vs_xgb']]
bars_d = ax2.bar(x, ens_df['delta_vs_xgb'], color=colors, alpha=0.85)
for bar, val in zip(bars_d, ens_df['delta_vs_xgb']):
    ypos = bar.get_height()+0.001 if val >= 0 else bar.get_height()-0.004
    ax2.text(bar.get_x()+bar.get_width()/2, ypos,
             f'{val:+.4f}', ha='center', fontsize=8)
ax2.axhline(0, color='black', lw=1, alpha=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(ens_df['endpoint'], rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('AUC gain (Ensemble − XGB alone)')
ax2.set_title('AUC improvement per endpoint', fontsize=11)
ax2.yaxis.grid(True, alpha=0.3)

plt.suptitle(f'Ensemble Results: {mean_xgb2:.4f} → {mean_ens:.4f} '
             f'({mean_ens-mean_xgb2:+.4f})', fontsize=12)
plt.tight_layout()
plt.savefig('plots/ensemble_results.png', dpi=150)
plt.show()
print('Saved → plots/ensemble_results.png')

# ROC curves
fig2, ax3 = plt.subplots(figsize=(9, 7))
for i, ep in enumerate(oof_probs):
    fpr, tpr, _ = roc_curve(oof_labels[ep], oof_probs[ep])
    row     = ens_df[ens_df['endpoint']==ep]
    ens_auc = float(row['ensemble_auc'].values[0]) if len(row)>0 else 0
    ax3.plot(fpr, tpr, color=COLORS[i % len(COLORS)], lw=1.5,
             label=f'{ep}  ({ens_auc:.3f})')
ax3.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title(f'ROC Curves — Ensemble\nMean AUC = {mean_ens:.4f}')
ax3.legend(loc='lower right', fontsize=8, ncol=2)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves_ensemble.png', dpi=150)
plt.show()
print('Saved → plots/roc_curves_ensemble.png')


---
## Save All Outputs

In [ ]:
with open('models/ensemble_models.pkl', 'wb') as f:
    pickle.dump(ensemble_models, f)
with open('models/xgb_models.pkl', 'wb') as f:
    pickle.dump(xgb_models, f)
if cat_models:
    with open('models/cat_models.pkl', 'wb') as f:
        pickle.dump(cat_models, f)
if brf_models:
    with open('models/brf_models.pkl', 'wb') as f:
        pickle.dump(brf_models, f)
with open('models/spw_dict.json', 'w') as f:
    json.dump(spw_dict, f, indent=2)

ens_df.to_csv('results/ensemble_results.csv', index=False)
thr_df.to_csv('results/optimal_thresholds.csv', index=False)
with open('models/optimal_thresholds.json', 'w') as f:
    json.dump(optimal_thresholds, f, indent=2)

best_ep  = ens_df.loc[ens_df['ensemble_auc'].idxmax()]
worst_ep = ens_df.loc[ens_df['ensemble_auc'].idxmin()]
most_imp = ens_df.loc[ens_df['delta_vs_xgb'].idxmax()]

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'  Fingerprint bits (after VT) : {fp_filtered_df.shape[1]:,}')
print(f'  Continuous features         : {cont_imputed.shape[1]:,}')
print(f'  TOTAL features              : {len(feat_cols):,}')
print()
print(f'  XGB alone mean AUC          : {mean_xgb2:.4f}')
print(f'  Ensemble mean AUC           : {mean_ens:.4f}')
print(f'  Improvement vs XGB          : {mean_ens - mean_xgb2:+.4f}')
print()
print(f'  Best endpoint  : {best_ep["endpoint"]}  (AUC={best_ep["ensemble_auc"]:.4f})')
print(f'  Worst endpoint : {worst_ep["endpoint"]}  (AUC={worst_ep["ensemble_auc"]:.4f})')
print(f'  Most improved  : {most_imp["endpoint"]}  (+{most_imp["delta_vs_xgb"]:.4f} vs XGB)')
print()
print('Saved:')
print('  models/ensemble_models.pkl')
print('  models/xgb_models.pkl')
print('  results/ensemble_results.csv')
print('  plots/ensemble_results.png')
print('  plots/roc_curves_ensemble.png')
print('  features_raw.csv')
print('  results/optimal_thresholds.csv')
print('  results/feature_importances.csv')
print('  plots/feature_importance_analysis.png')
print('  plots/prediction_profile.png')
print()
print('To predict a new molecule:')
print("  with open('models/ensemble_models.pkl','rb') as f:")
print("      models = pickle.load(f)")
print("  ep_models = models['NR-AR']")
print("  w = ep_models['weights']")
print("  p_xgb = ep_models['xgb'].predict_proba(X_new)[:,1]")
print("  p_lgb = ep_models['lgb'].predict_proba(X_new)[:,1] if ep_models['lgb'] else p_xgb")
print("  p_rf  = ep_models['rf'].predict_proba(X_new)[:,1]")
print("  prob  = w[0]*p_xgb + w[1]*p_lgb + w[2]*p_rf")
